# NL_MSc_halozat – Debreceni Egyetem: publikációs együttműködések, 2015–2024

**Munkajegyzet / számítási notebook.** Scopus-alapú kari publikációs exportok, DE-s szerzőtörzs, NetworkX-elemzések és Gephi-kimenetek. A gráfok **együttműködési és társszerzői**, nem idézési hálózatok.

A vizsgált állomány a rendelkezésre álló kari exportokból származik; nem a teljes DE-publikációs korpusz. A referenciafutásban 8 223 egyedi publikáció és 7 288 szerzői entitás szerepel. Az adathozzáférés és a publikálhatóság korlátait a README rögzíti.

**Értelmezési szabályok**

- Publikációkulcs: **EID**, hiányában DOI, végül forrássor-azonosító. A belső `pub_id` csak a feldolgozott mintán belül érvényes.
- DE-szerző: egyezés a referencia-szerzőtörzs **Scopus Author ID** mezőjével; a névegyezés önmagában nem azonosítás.
- A szerző–kar hozzárendelés három konzervatív evidenciakörből áll. A bizonytalan profilokat nem olvasztjuk automatikusan össze.
- A ROR-os intézmény–ország azonosítás **nem** helyettesíti a közvetlen publikáció–ország aggregációt.
- A szerzőháló betweenness-mutatója mintavételes közelítés, a kari hálóé egzakt. A Gephi a megjelenítés eszköze.

**Futtatás:** a projekt gyökérkönyvtárából, a README-ben rögzített Python-környezettel. A részletes auditok helyben maradnak; a nyers és szerzőszintű exportok nem nyilvános adatok.


In [ ]:
# Könyvtárak a beolvasáshoz és a táblakezeléshez

import csv

import numpy as np
import pandas as pd


In [ ]:
CSV_ENCODING = "utf-8-sig"
CSV_SEP = ";"
CSV_QUOTING = csv.QUOTE_ALL
SAVE_INDEX = False

DE_CENTRAL_AFFIL_ID = "60004400"


In [ ]:
# A futtatás a projekt gyökeréből indul.

from pathlib import Path
import sys

ROOT = Path(".").resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
# A feldolgozófüggvények a utils csomagban vannak.
from utils.io_utils import (
    ensure_directories,
    discover_project_inputs,
    read_kari_excel,
    build_publication_master,
    build_publication_faculty_link,
    build_faculty_dict,
    read_de_author_excel,
    build_de_author_ref,
    build_publication_author_occurrence,
)

from utils.parse_utils import normalize_doi_series, normalize_eid

from utils.author_merge_utils import (
    flag_de_author_occurrences,
    build_author_kar_evidence_round1,
    build_author_kar_evidence_round2,
    build_author_kar_evidence_round3,
    build_author_kar_profile,
    build_author_entities,
    build_author_entity_scopus_id_tables,
    build_publication_author_entity_link,
)

from utils.affiliation_utils import (
    build_publication_affiliation_occurrence,
    build_de_affiliation_dict,
    build_publication_external_root_occurrence,
    build_publication_institution_link,
    build_publication_country_occurrence,
    infer_institution_countries,
)

from utils.network_utils import (
    build_author_author_edges,
    build_author_nodes_export,
    build_faculty_faculty_edges,
    build_faculty_nodes_export,
    build_de_institution_edges,
    build_country_aggregations,
    compute_faculty_network_metrics,
    compute_author_network_metrics,
    build_gephi_exports,

)

from utils.analysis_utils import (
    prepare_author_meta_analysis_base,
    build_faculty_meta_summary,
    build_author_nodes_gephi_kat
)


In [ ]:
# Helyi forrás-, cache- és kimeneti könyvtárak.
SOURCE = ROOT / "source_data"
SOURCE_KARI = SOURCE / "kari"
UTILS = ROOT / "utils"

DATA_OUT = ROOT / "data_out"
INTERIM = DATA_OUT / "interim"
PROCESSED = DATA_OUT / "processed"
EXPORT = DATA_OUT / "export"
FIGURES = ROOT / "figures"

ensure_directories([DATA_OUT, INTERIM, PROCESSED, EXPORT, FIGURES])

if str(UTILS.resolve()) not in sys.path:
    sys.path.append(str(UTILS.resolve()))

print("Projekt gyökér:", ROOT.resolve())
print("Forrásmappa:", SOURCE.resolve())
print("Kari forrásmappa:", SOURCE_KARI.resolve())
print("Utils mappa:", UTILS.resolve())
print("Kimeneti mappák létrehozva.")


In [ ]:
from joblib import Memory

COMPUTE_CACHE = Memory(
    location=str(INTERIM / "joblib_cache"),
    verbose=0
)


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style, apply openpyxl's default"
)


In [ ]:
# Kari rövidítések

KAR_MAP = {
    "Altalanos_Orvostudomanyi_Kar": "ÁOK",
    "Agrar_Kutatointezetek_es_Tangazdasag": "AKT",
    "Informatikai_Kar": "IK",
    "Allam-_es_Jogtudomanyi_Kar": "ÁJK",
    "Bolcseszettudomanyi_Kar": "BTK",
    "Egeszsegugyi_Kar": "EÜK",
    "Fogorvostudomanyi_Kar": "FOK",
    "Gazdasagtudomanyi_Kar": "GTK",
    "Gyogyszeresztudomanyi_Kar": "GyTK",
    "Klinikai_Kozpont": "KK",
    "Klinikai_Kozpont_Kenezy_Gyula_Campus": "KK-KGyC",
    "Mezogazdasag-,_Elelmiszertudomanyi_es_Kornyezetgazdalkodasi_Kar": "MÉKK",
    "Muszaki_Kar": "MK",
    "Nepegeszsegugyi_Kar": "NEK",
    "Termeszettudomanyi_es_Technologiai_Kar": "TTK",
    "Zenemuveszeti_Kar": "ZMK",
    "Gyermeknevelesi_es_Felnottkepzesi_Kar": "GYFK",
}

KAR_CODES = sorted(KAR_MAP.values())
KAR_CODES


In [ ]:
kari_files, author_candidates = discover_project_inputs(
    source_dir=SOURCE,
    source_kari_dir=SOURCE_KARI
)

print("Kari Excel fájlok száma:", len(kari_files))
for p in kari_files:
    print(" -", p.name)

print("\nDE szerzőlista fájl(ok):")
for p in author_candidates:
    print(" -", p.name)


In [ ]:
# A bemenetekből egy szerzőlista szükséges; hiba esetén azonnal jelezzünk.
assert kari_files, "Nem található kari Scopus Excel-export a source_data/kari mappában"
assert author_candidates, "Nem található DE_confirmed_authors*.xlsx a source_data mappában"
AUTHOR_FILE = author_candidates[0]
print("DE szerzőlista:", AUTHOR_FILE.name, "| kari fájlok:", len(kari_files))


In [ ]:
# A más fejezetekben hivatkozott azonosítóoszlopok
PUB_ID_COL = "pub_id"
AUTHOR_ID_COL = "author_id"


In [ ]:
# Használt állományútvonalak; a nem hivatkozott exportneveket kihagyjuk.

PATHS = {
    "publication_master": PROCESSED / "publication_master.csv",
    "publication_faculty_link": PROCESSED / "publication_faculty_link.csv",
    "faculty_dict": PROCESSED / "faculty_dict.csv",
    "de_author_ref": PROCESSED / "de_author_ref.csv",
    "publication_author_occurrence": PROCESSED / "publication_author_occurrence.csv",
    "publication_author_occurrence_deflag": PROCESSED / "publication_author_occurrence_deflag.csv",
    "de_affiliation_dict": PROCESSED / "de_affiliation_dict.csv",
    "institution_dict": PROCESSED / "institution_dict.csv",
    "publication_institution_link": PROCESSED / "publication_institution_link.csv",
    "publication_author_entity_link": PROCESSED / "publication_author_entity_link.csv",
    "author_nodes_export": EXPORT / "author_nodes_export.csv",
    "author_edges_export": EXPORT / "author_edges_export.csv",
    "faculty_nodes_export": EXPORT / "faculty_nodes_export.csv",
    "faculty_edges_export": EXPORT / "faculty_edges_export.csv",
}

pd.Series({k: str(v) for k, v in PATHS.items()})


In [ ]:
import os
import csv
import time
import psutil
import threading
from datetime import datetime

RAM_LOG = INTERIM / f"ram_monitor_{datetime.now():%Y%m%d_%H%M%S}.csv"
_ram_process = psutil.Process(os.getpid())
_ram_stop = threading.Event()
_ram_state = {"cell": "start"}

def _ram_mark_cell(info):
    lines = info.raw_cell.strip().splitlines()
    _ram_state["cell"] = lines[0][:100] if lines else "empty"

get_ipython().events.register("pre_run_cell", _ram_mark_cell)

def _ram_monitor():
    with open(RAM_LOG, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["time", "kernel_ram_gb", "available_ram_gb", "cell"])

        while not _ram_stop.is_set():
            writer.writerow([
                datetime.now().isoformat(timespec="seconds"),
                round(_ram_process.memory_info().rss / 1024**3, 3),
                round(psutil.virtual_memory().available / 1024**3, 3),
                _ram_state["cell"]
            ])
            f.flush()
            _ram_stop.wait(2)

threading.Thread(target=_ram_monitor, daemon=True).start()

print("Memórianapló:", RAM_LOG)


## Bemenetek és könyvtárak

- `source_data/kari/Publications_of*.xlsx`: kari Scopus-exportok (17 szervezeti egység).
- `source_data/DE_confirmed_authors*.xlsx`: DE-szerzők referenciaállománya.
- `utils/`: beolvasás, szöveg- és entitásnormalizálás, intézményfeloldás, hálózati számítások.
- `data_out/interim/`: köztes auditok, joblib és ROR-cache; `data_out/processed/`: eredménytáblák, Gephi-exportok, ábrák.

**Figyelni kell:** a munkakönyvtár legyen a projekt gyökere. Az eredeti Excel-fájlokat és részletes szcientometriai adatokat nem szabad automatikusan nyilvánossá tenni.


# 1. Kari forrásfájlok

A fejléc nem feltétlenül az első sorban van; az `io_utils` keresi meg a szükséges oszlopok alapján. A kari kontextus (`source_kar`, `source_file`) minden forrássorhoz megmarad. Az összefűzött `df_raw_faculty_publications` **még nem deduplikált**. A fájlszintű eltérések a `df_kari_file_audit` táblában követhetők.


In [ ]:
# Bemeneti fájlok és elérhetőség (a felderítés már a 0. fejezetben megtörtént).
assert SOURCE.is_dir() and SOURCE_KARI.is_dir(), "Hiányzó source_data vagy source_data/kari mappa"
print("Kari exportok:", len(kari_files), "| szerzőlisták:", len(author_candidates))


In [ ]:
kari_frames = []
kari_audit_rows = []

for xlsx_path in kari_files:
    print("Beolvasás alatt:", xlsx_path.name)
    df_kari, audit_row = read_kari_excel(xlsx_path, kar_map=KAR_MAP)
    kari_frames.append(df_kari)
    kari_audit_rows.append(audit_row)

df_kari_file_audit = pd.DataFrame(kari_audit_rows)

print("\nBeolvasott DataFrame-ek száma:", len(kari_frames))
print("Audit tábla alakja:", df_kari_file_audit.shape)


In [ ]:
# Audit tábla áttekintése

df_kari_file_audit.sort_values(["source_kar", "source_file"]).reset_index(drop=True)


In [ ]:
# Az összes kari rekord összefűzése egy közös nyers táblába

df_raw_faculty_publications = pd.concat(kari_frames, ignore_index=True, sort=False)

print("Egyesített nyers kari publikációs tábla alakja:", df_raw_faculty_publications.shape)


In [ ]:
# Gyors oszlopellenőrzés

raw_columns = pd.DataFrame({
    "column_name": df_raw_faculty_publications.columns
})

print("Oszlopok száma:", len(raw_columns))
raw_columns


In [ ]:
# Kulcsoszlopok jelenlétének ellenőrzése

expected_core_columns = [
    "Title",
    "DOI",
    "EID",
    "Authors",
    "Scopus Author Ids",
    "Scopus Affiliation IDs",
    "Scopus Affiliation names",
    "Year",
    "Citations",
    "Field-Weighted Citation Impact",
]

core_column_check = pd.DataFrame({
    "column_name": expected_core_columns,
    "present": [col in df_raw_faculty_publications.columns for col in expected_core_columns]
})

core_column_check


In [ ]:
# Alapminták megtekintése

display_cols = [
    col for col in [
        "Title",
        "DOI",
        "EID",
        "Authors",
        "Scopus Author Ids",
        "Scopus Affiliation IDs",
        "Scopus Affiliation names",
        "Year",
        "source_kar",
        "source_file",
    ]
    if col in df_raw_faculty_publications.columns
]

df_raw_faculty_publications[display_cols].head(5)


In [ ]:
# Egyszerű alapstatisztikák a nyers összefűzött táblára

raw_summary = {
    "n_rows": len(df_raw_faculty_publications),
    "n_cols": len(df_raw_faculty_publications.columns),
    "n_source_files": df_raw_faculty_publications["source_file"].nunique(),
    "n_source_kars": df_raw_faculty_publications["source_kar"].nunique(),
}

pd.Series(raw_summary)


In [ ]:
# A fájlszintű beolvasási audit mentése; a nyers kari tábla duplikált CSV-másolata nem kell.
KARI_FILE_AUDIT_PATH = INTERIM / "kari_file_audit.csv"

df_kari_file_audit.to_csv(
    KARI_FILE_AUDIT_PATH,
    index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING
)
print("Mentve:", KARI_FILE_AUDIT_PATH)


# 2. Publikációs törzs – EID-elsődleges deduplikáció

Kulcssorrend: `EID` → normalizált `DOI` → forrássor. Azonos kulcsú kari rekordok egyetlen `publication_master` sorhoz tartoznak; az eredeti előfordulások `df_raw_faculty_publications_keyed` táblája a közös `pub_id`-t kapja. Az EID/DOI hiány és a többkarú előfordulás külön auditálható. **A DOI nem elsődleges kulcs.**


In [ ]:
# A publikációs törzsben megtartandó főbb mezők

PUB_MASTER_KEEP_COLS = [
    "DOI",
    "EID",
    "Title",
    "Year",
    "Citations",
    "Field-Weighted Citation Impact",
    "Number of Authors",
    "Number of Institutions",
    "Number of Countries/Regions",
]


In [ ]:
# DOI / EID kulcsaudit
# DOI-hiány megengedett: az EID az elsődleges publikációkulcs.
# DOI csak hiányzó EID esetén használatos.

key_audit_df = df_raw_faculty_publications.copy()
key_audit_df["doi_norm"] = normalize_doi_series(key_audit_df["DOI"])

if "EID" in key_audit_df.columns:
    key_audit_df["eid_norm"] = key_audit_df["EID"].map(normalize_eid)
else:
    key_audit_df["eid_norm"] = np.nan

key_audit = pd.Series({
    "n_rows": len(key_audit_df),
    "n_missing_doi_norm": int(key_audit_df["doi_norm"].isna().sum()),
    "n_missing_eid_norm": int(key_audit_df["eid_norm"].isna().sum()),
    "n_missing_doi_but_has_eid": int(
        key_audit_df["doi_norm"].isna().mul(key_audit_df["eid_norm"].notna()).sum()
    ),
    "n_missing_both_doi_and_eid": int(
        key_audit_df["doi_norm"].isna().mul(key_audit_df["eid_norm"].isna()).sum()
    ),
})

key_audit


In [ ]:
# Publikációs törzs és nyers→pub_id visszakötés előállítása

publication_master, df_raw_faculty_publications_keyed = build_publication_master(
    df_raw=df_raw_faculty_publications,
    eid_col="EID",
    doi_col="DOI",
    keep_cols=PUB_MASTER_KEEP_COLS,
    source_kar_col="source_kar",
    source_file_col="source_file",
    pub_id_prefix="PUB"
)

print("publication_master alakja:", publication_master.shape)
print("df_raw_faculty_publications_keyed alakja:", df_raw_faculty_publications_keyed.shape)


In [ ]:
# Gyors ellenőrzés: minden nyers rekord kapott-e pub_id-t?

df_raw_faculty_publications_keyed["pub_id"].isna().sum()


In [ ]:
# Publikációs törzs első sorai

publication_master.head(10)


In [ ]:
# Duplikációs audit: hány nyers kari rekord olvad össze ugyanabba a publikációba?

publication_master["n_source_rows"].describe()


In [ ]:
# Mely publikációk szerepeltek a legtöbb kari forrásban?

publication_master.sort_values(
    ["n_source_kars", "n_source_rows", "Year"],
    ascending=[False, False, False]
).head(15)


In [ ]:
# Ellenőrzés: egy DOI pontosan egy pub_id-hoz tartozik-e?

doi_pub_check = (
    df_raw_faculty_publications_keyed
    .groupby("doi_norm")["pub_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

doi_pub_check


In [ ]:
# DOI duplikációs összkép

publication_dedup_summary = pd.Series({
    "n_raw_rows": len(df_raw_faculty_publications),
    "n_unique_publications": len(publication_master),
    "n_duplicate_rows_removed": len(df_raw_faculty_publications) - len(publication_master),
    "mean_source_rows_per_publication": publication_master["n_source_rows"].mean(),
    "mean_source_kars_per_publication": publication_master["n_source_kars"].mean(),
})

publication_dedup_summary


In [ ]:
# A publikációs törzs marad tartós export; a teljes kulcsolt nyers táblát a memória őrzi.
publication_master.to_csv(
    PATHS["publication_master"],
    index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING
)
print("Mentve:", PATHS["publication_master"])


# 3. Publikáció–kar hozzárendelés

Egy publikáció több kari forrásban is előfordulhat. A `publication_faculty_link` egyedi `(pub_id, kar_code)` párokat tartalmaz; ebből származik a kari lefedettség és a `faculty_dict`. A többszörös kari megjelenés együttműködési kapcsolatra utalhat, de a forráslefedettség sajátosságait is figyelembe kell venni.


In [ ]:
# Publikáció–kar kapcsolatok előállítása

publication_faculty_link = build_publication_faculty_link(
    df_raw_keyed=df_raw_faculty_publications_keyed,
    pub_id_col="pub_id",
    source_kar_col="source_kar",
    source_file_col="source_file",
)

faculty_dict = build_faculty_dict(
    kar_map=KAR_MAP,
    publication_faculty_link=publication_faculty_link,
)

print("publication_faculty_link alakja:", publication_faculty_link.shape)
print("faculty_dict alakja:", faculty_dict.shape)


In [ ]:
# Első sorok áttekintése

publication_faculty_link.head(10)


In [ ]:
# Kar-törzs áttekintése

faculty_dict


In [ ]:
# Ellenőrzés: minden publikáció kapott-e legalább egy kari kapcsolatot?

n_publications_master = publication_master["pub_id"].nunique()
n_publications_with_kar = publication_faculty_link["pub_id"].nunique()

pd.Series({
    "n_publications_master": n_publications_master,
    "n_publications_with_kar": n_publications_with_kar,
    "n_missing_publication_links": n_publications_master - n_publications_with_kar,
})


In [ ]:
# Hány karhoz kötődik egy publikáció?

pub_kar_counts = (
    publication_faculty_link
    .groupby("pub_id")["kar_code"]
    .nunique()
    .reset_index(name="n_kars")
)

pub_kar_counts["n_kars"].describe()


In [ ]:
# Többkarú publikációk mintái

multi_kar_publications = (
    pub_kar_counts
    .query("n_kars > 1")
    .sort_values("n_kars", ascending=False)
)

multi_kar_publications.head(15)


In [ ]:
# Mely karokhoz hány publikáció tartozik?

faculty_dict.sort_values(["publication_count", "kar_code"], ascending=[False, True]).reset_index(drop=True)


In [ ]:
# A törzstábla és a publikáció–kar kapcsolatok kari darabszáma egyezzen.

pub_kar_compare = (
    publication_master[["pub_id", "n_source_kars"]]
    .merge(
        pub_kar_counts.rename(columns={"n_kars": "n_kars_from_link"}),
        on="pub_id",
        how="left"
    )
)

pub_kar_compare["n_kars_from_link"] = pub_kar_compare["n_kars_from_link"].fillna(0).astype(int)

(pub_kar_compare["n_source_kars"] == pub_kar_compare["n_kars_from_link"]).value_counts(dropna=False)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

publication_faculty_link.to_csv(
    PATHS["publication_faculty_link"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_dict.to_csv(
    PATHS["faculty_dict"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["publication_faculty_link"])
print(" -", PATHS["faculty_dict"])


# 4. DE-s szerzőtörzs

Az Excel fejlécének, üres zárósorainak és oszlopneveinek egységesítése után a `de_author_ref` őrzi meg a Scopus Author ID-ket és a releváns bibliometriai mezőket. A h-index, FWCI és idézettségi adatok **forrásprofilhoz kötöttek**; későbbi szerzőösszevonáskor nem összegezhetők.


In [ ]:
# DE szerzőlista beolvasása

df_de_author_raw, author_audit = read_de_author_excel(AUTHOR_FILE)
df_de_author_audit = pd.DataFrame([author_audit])

print("df_de_author_raw alakja:", df_de_author_raw.shape)
print("df_de_author_audit alakja:", df_de_author_audit.shape)


In [ ]:
# Audit áttekintése

df_de_author_audit


In [ ]:
# Nyers oszlopok áttekintése

pd.DataFrame({"column_name": df_de_author_raw.columns})


In [ ]:
# A tisztított DE szerző-referencia tábla előállítása

de_author_ref = build_de_author_ref(df_de_author_raw)

print("de_author_ref alakja:", de_author_ref.shape)


In [ ]:
# Első sorok megtekintése

de_author_ref.head(10)
de_author_ref.dtypes


In [ ]:
# Ellenőrzés: kötelező mezők jelenléte

required_author_cols = [
    "ref_author_name",
    "ref_scopus_id",
    "citations",
    "cit_per_pub",
    "fwci",
    "h_index",
]

pd.DataFrame({
    "column_name": required_author_cols,
    "present": [col in de_author_ref.columns for col in required_author_cols]
})


In [ ]:
# Ellenőrzés: van-e hiányzó név vagy Scopus ID?

pd.Series({
    "n_rows": len(de_author_ref),
    "missing_ref_author_name": int(de_author_ref["ref_author_name"].isna().sum()),
    "missing_ref_scopus_id": int(de_author_ref["ref_scopus_id"].isna().sum()),
    "duplicate_name_scopus_pairs": int(
        de_author_ref.duplicated(subset=["ref_author_name", "ref_scopus_id"]).sum()
    ),
    "unique_scopus_ids": int(de_author_ref["ref_scopus_id"].nunique()),
})


In [ ]:
# Azonos névhez rendelt különböző Scopus ID-k – összevonási diagnosztika.

name_id_counts = (
    de_author_ref
    .groupby("ref_author_name")["ref_scopus_id"]
    .nunique()
    .reset_index(name="n_scopus_ids")
    .sort_values("n_scopus_ids", ascending=False)
)

name_id_counts.head(15)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

de_author_ref.to_csv(
    PATHS["de_author_ref"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["de_author_ref"])


# 5. Szerző-előfordulások

Az `Authors` és a `Scopus Author Ids` listák pozíció alapján párosulnak, publikációs és kari kontextussal. A különböző hosszúságú listákat az audit jelzi; a hiányzó azonosítót nem pótoljuk találgatással. A szerzőszámok és a ténylegesen kinyert sorok közötti eltérés külön ellenőrzést kap.


In [ ]:
# Szerző-előfordulások előállítása

publication_author_occurrence, author_occurrence_audit = build_publication_author_occurrence(
    df_raw_keyed=df_raw_faculty_publications_keyed,
    pub_id_col="pub_id",
    authors_col="Authors",
    author_ids_col="Scopus Author Ids",
    source_kar_col="source_kar",
    source_file_col="source_file",
)

print("publication_author_occurrence alakja:", publication_author_occurrence.shape)


In [ ]:
# Audit

author_occurrence_audit


In [ ]:
# Ellenőrzés: minden publikációhoz van-e szerző-előfordulás?

pd.Series({
    "n_publications_master": int(publication_master["pub_id"].nunique()),
    "n_publications_in_occurrence": int(publication_author_occurrence["pub_id"].nunique()),
    "n_missing_publications": int(
        publication_master["pub_id"].nunique() - publication_author_occurrence["pub_id"].nunique()
    ),
})


In [ ]:
# Ellenőrzés: pozíciók eloszlása

publication_author_occurrence["author_position"].describe()


In [ ]:
# Ellenőrzés: hány szerző szerepel publikációnként és kari kontextusonként?

pub_author_counts = (
    publication_author_occurrence
    .groupby(["pub_id", "source_kar"])["author_position"]
    .max()
    .reset_index(name="n_authors_from_occurrence")
)

pub_author_counts["n_authors_from_occurrence"].describe()


In [ ]:
# Véletlen minta ellenőrzésre

publication_author_occurrence.sample(10, random_state=42)


In [ ]:
pd.Series({
    "missing_author_name_raw": int(publication_author_occurrence["author_name_raw"].isna().sum()),
    "missing_scopus_author_id": int(publication_author_occurrence["scopus_author_id"].isna().sum()),
})


In [ ]:
raw_author_count_check = (
    df_raw_faculty_publications_keyed[
        ["pub_id", "source_kar", "source_file", "Number of Authors"]
    ]
    .drop_duplicates()
    .rename(columns={"Number of Authors": "n_authors_raw"})
    .copy()
)

raw_author_count_check["n_authors_raw"] = pd.to_numeric(
    raw_author_count_check["n_authors_raw"],
    errors="coerce"
).astype("Int64")

occ_author_count_check = (
    publication_author_occurrence
    .groupby(["pub_id", "source_kar", "source_file"])["author_position"]
    .max()
    .reset_index(name="n_authors_occ")
)

author_count_compare = raw_author_count_check.merge(
    occ_author_count_check,
    on=["pub_id", "source_kar", "source_file"],
    how="left"
)

author_count_compare["match"] = (
    author_count_compare["n_authors_raw"] == author_count_compare["n_authors_occ"]
)

author_count_compare["match"].value_counts(dropna=False)


In [ ]:
author_count_compare.loc[~author_count_compare["match"]].head(20)


In [ ]:
publication_author_occurrence.duplicated(
    subset=["pub_id", "source_kar", "author_position", "author_name_raw", "scopus_author_id"]
).sum()


In [ ]:
author_count_compare["diff"] = (
    author_count_compare["n_authors_occ"] - author_count_compare["n_authors_raw"]
)

author_count_compare.loc[~author_count_compare["match"], "diff"].value_counts().sort_index()


In [ ]:
author_count_compare.loc[
    ~author_count_compare["match"],
    ["pub_id", "source_kar", "source_file", "n_authors_raw", "n_authors_occ", "diff"]
].head(20)


In [ ]:
bad_pub_ids = author_count_compare.loc[~author_count_compare["match"], "pub_id"].head(5).tolist()

df_raw_faculty_publications_keyed.loc[
    df_raw_faculty_publications_keyed["pub_id"].isin(bad_pub_ids),
    ["pub_id", "source_kar", "Title", "Authors", "Scopus Author Ids", "Number of Authors"]
]


In [ ]:
author_count_compare.loc[~author_count_compare["match"], "n_authors_raw"].describe()


In [ ]:
df_raw_faculty_publications_keyed["doi_norm"].isna().sum()


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

publication_author_occurrence.to_csv(
    PATHS["publication_author_occurrence"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["publication_author_occurrence"])


In [ ]:
bad_pubs = (
    df_raw_faculty_publications_keyed
    .groupby("pub_id")
    .agg(
        n_titles=("Title", "nunique"),
        n_eids=("EID", "nunique"),
        n_dois=("doi_norm", "nunique"),
        n_rows=("Title", "size"),
        source_kars=("source_kar", "nunique"),
    )
    .sort_values(["n_titles", "n_eids", "n_dois", "n_rows"], ascending=[False, False, False, False])
)

suspicious_pubs = bad_pubs.loc[
    (bad_pubs["n_titles"] > 1) |
    (bad_pubs["n_eids"] > 1) |
    (bad_pubs["n_dois"] > 1)
].copy()

print("Összes pub_id:", len(bad_pubs))
print("Gyanús pub_id-k száma:", len(suspicious_pubs))

suspicious_pubs.head(30)


In [ ]:
publication_master["match_key_type"].value_counts(dropna=False)


# 6. DE-szerzők jelölése

Biztos DE-match kizárólag a szerzőtörzsben megtalálható **Scopus Author ID** alapján. Az egyező név, de eltérő/hiányzó azonosító csak diagnosztikai jel; nem növeli automatikusan a DE-szerzők számát. Fő kimenet: `publication_author_occurrence_deflag`.


In [ ]:
publication_author_occurrence_deflag, de_author_match_audit = flag_de_author_occurrences(
    publication_author_occurrence=publication_author_occurrence,
    de_author_ref=de_author_ref,
    occurrence_id_col="scopus_author_id",
    ref_id_col="ref_scopus_id",
    occurrence_name_col="author_name_raw",
    ref_name_col="ref_author_name",
)

print("publication_author_occurrence_deflag alakja:", publication_author_occurrence_deflag.shape)


In [ ]:
de_author_match_audit


In [ ]:
publication_author_occurrence_deflag.head(10)


In [ ]:
pd.Series({
    "n_total_occurrence_rows": len(publication_author_occurrence_deflag),
    "n_de_rows": int(publication_author_occurrence_deflag["is_de_author_ref_match"].sum()),
    "n_non_de_rows": int((~publication_author_occurrence_deflag["is_de_author_ref_match"]).sum()),
    "de_row_ratio": float(publication_author_occurrence_deflag["is_de_author_ref_match"].mean()),
})


In [ ]:
# Hány egyedi DE Scopus ID jelent meg az occurrence táblában?

pd.Series({
    "unique_ref_scopus_ids_total": int(de_author_ref["ref_scopus_id"].nunique()),
    "unique_ref_scopus_ids_matched": int(
        publication_author_occurrence_deflag.loc[
            publication_author_occurrence_deflag["is_de_author_ref_match"],
            "scopus_author_id"
        ].nunique()
    ),
})


In [ ]:
# Diagnosztika: vannak-e olyan unmatched sorok, ahol a név pontosan szerepel a DE törzsben?

publication_author_occurrence_deflag.loc[
    (~publication_author_occurrence_deflag["is_de_author_ref_match"]) &
    (publication_author_occurrence_deflag["has_exact_name_in_ref"]),
    ["pub_id", "source_kar", "author_name_raw", "scopus_author_id", "has_exact_name_in_ref"]
].head(20)


In [ ]:
# Mely karokban mennyi DE-s occurrence van?

de_occ_by_kar = (
    publication_author_occurrence_deflag
    .groupby("source_kar")["is_de_author_ref_match"]
    .sum()
    .reset_index(name="n_de_occurrence_rows")
    .sort_values("n_de_occurrence_rows", ascending=False)
)

de_occ_by_kar


In [ ]:
# Véletlen minta DE-s sorokból

publication_author_occurrence_deflag.loc[
    publication_author_occurrence_deflag["is_de_author_ref_match"]
].sample(min(10, int(publication_author_occurrence_deflag["is_de_author_ref_match"].sum())), random_state=42)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

publication_author_occurrence_deflag.to_csv(
    PATHS["publication_author_occurrence_deflag"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["publication_author_occurrence_deflag"])


# 7. Kari affiliációk

Az affiliációs ID-k és nevek párjait külön előfordulástáblába bontjuk. A központi DE-ID (`60004400`) kizárása után karonként a **legnagyobb publikációs lefedettségű** affiliációs ID kerül a `de_affiliation_dict` táblába. A három legjobb jelölt auditja segít a határesetek ellenőrzésében; ez nem puszta névegyezés.


In [ ]:
publication_affiliation_occurrence, affiliation_occurrence_audit = build_publication_affiliation_occurrence(
    df_raw_keyed=df_raw_faculty_publications_keyed,
    pub_id_col="pub_id",
    affil_ids_col="Scopus Affiliation IDs",
    affil_names_col="Scopus Affiliation names",
    source_kar_col="source_kar",
    source_file_col="source_file",
)

print("publication_affiliation_occurrence alakja:", publication_affiliation_occurrence.shape)


In [ ]:
affiliation_occurrence_audit


In [ ]:
de_affiliation_dict, de_affiliation_audit, de_affiliation_candidates_top3 = build_de_affiliation_dict(
    publication_affiliation_occurrence=publication_affiliation_occurrence,
    de_central_affil_id=DE_CENTRAL_AFFIL_ID,
    source_kar_col="source_kar",
    affil_id_col="affil_id",
    affil_name_col="affil_name_raw",
)

print("de_affiliation_dict alakja:", de_affiliation_dict.shape)
print("de_affiliation_candidates_top3 alakja:", de_affiliation_candidates_top3.shape)


In [ ]:
de_affiliation_dict.head(20)


In [ ]:
# Ellenőrzés: hány kart fed le a DE affiliációs szótár?

pd.Series({
    "n_expected_kars": len(KAR_MAP),
    "n_kars_in_dict": int(de_affiliation_dict["kar_code"].nunique()),
    "n_missing_kars": len(KAR_MAP) - int(de_affiliation_dict["kar_code"].nunique()),
})


In [ ]:
# Mely karokhoz hány affiliációs ID tartozik?

de_affiliation_dict["kar_code"].value_counts().sort_index()


In [ ]:
# Van-e konfliktusos affiliációs ID?

de_affiliation_candidates_top3.head(30)


In [ ]:
de_affiliation_candidates_top3.sort_values(
    ["kar_code", "rank_within_kar"],
    ascending=[True, True]
).reset_index(drop=True)


In [ ]:
pd.Series({
    "n_rows": len(de_affiliation_dict),
    "n_unique_kars": de_affiliation_dict["kar_code"].nunique(),
    "n_unique_affil_ids": de_affiliation_dict["affil_id"].nunique(),
})


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AFFILIATION_OCCURRENCE_PATH = INTERIM / "publication_affiliation_occurrence.csv"
DE_AFFILIATION_CANDIDATES_PATH = INTERIM / "de_affiliation_candidates_top3.csv"

publication_affiliation_occurrence.to_csv(
    AFFILIATION_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

de_affiliation_dict.to_csv(
    PATHS["de_affiliation_dict"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

de_affiliation_candidates_top3.to_csv(
    DE_AFFILIATION_CANDIDATES_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING

)

print("Mentve:")
print(" -", AFFILIATION_OCCURRENCE_PATH)
print(" -", PATHS["de_affiliation_dict"])
print(" -", DE_AFFILIATION_CANDIDATES_PATH)


# 8. Külső intézményi gyökérjelöltek

A kari/központi DE-affiliációk kizárása után az affiliációs lista sorrendjét és az intézmény-, illetve alegységjellegű neveket használjuk. Ha nincs felismerhető intézményi gyökér, az első külső affiliáció marad tartalékként. A heurisztikus `root_affil_id` **jelölt**, nem automatikusan hitelesített intézményhierarchia.


In [ ]:
cached_external_roots = COMPUTE_CACHE.cache(
    build_publication_external_root_occurrence
)

publication_external_root_occurrence, external_root_audit, external_root_affil_stats = cached_external_roots(
    publication_affiliation_occurrence=publication_affiliation_occurrence,
    de_affiliation_dict=de_affiliation_dict,
    de_central_affil_id=DE_CENTRAL_AFFIL_ID,
    pub_id_col="pub_id",
    affil_pos_col="affil_position",
    affil_id_col="affil_id",
    affil_name_col="affil_name_raw",
)

print("publication_external_root_occurrence alakja:", publication_external_root_occurrence.shape)
print("external_root_affil_stats alakja:", external_root_affil_stats.shape)


In [ ]:
external_root_audit


In [ ]:
publication_external_root_occurrence.head(20)


In [ ]:
external_root_affil_stats.head(30)


In [ ]:
# Hány publikációban van legalább 1 külső intézményi root?

pd.Series({
    "n_publications_total": int(publication_master["pub_id"].nunique()),
    "n_publications_with_external_roots": int(publication_external_root_occurrence["pub_id"].nunique()),
    "share_with_external_roots": float(
        publication_external_root_occurrence["pub_id"].nunique() / publication_master["pub_id"].nunique()
    ) if len(publication_master) > 0 else np.nan,
})


In [ ]:
# Mely kiválasztások jöttek fallbackből?

publication_external_root_occurrence["selection_reason"].value_counts(dropna=False)


In [ ]:
# Véletlen minta ellenőrzésre

publication_external_root_occurrence.sample(
    min(15, len(publication_external_root_occurrence)),
    random_state=42
)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

PUBLICATION_EXTERNAL_ROOT_PATH = INTERIM / "publication_external_root_occurrence.csv"
EXTERNAL_ROOT_AFFIL_STATS_PATH = INTERIM / "external_root_affil_stats.csv"

publication_external_root_occurrence.to_csv(
    PUBLICATION_EXTERNAL_ROOT_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

external_root_affil_stats.to_csv(
    EXTERNAL_ROOT_AFFIL_STATS_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PUBLICATION_EXTERNAL_ROOT_PATH)
print(" -", EXTERNAL_ROOT_AFFIL_STATS_PATH)


# 9. Publikáció–intézmény kapcsolatok és országfeloldás

**9/A–B.** A külső gyökér-ID-kből publikáció–intézmény kapcsolat, valamint külön, közvetlen `Country/Region` alapú publikáció–ország tábla készül.

**9/C.** ROR v2 `affiliation` lekérdezés, folytatható helyi cache-sel: `data_out/interim/ror_match_cache.csv`. A cache intézményazonosítóhoz **és névhez** kötött. Az API-adatbázis időben változhat, ezért az eredeti cache megőrzése szükséges az azonos eredményhez.

**9/D.** `chosen=True` ROR-találat országa csak akkor elfogadható, ha az intézmény valamennyi vizsgált publikációjának országlistájában szerepel. Ellentmondás → `ror_country_conflict_review`, ország nélkül. Tartalék besorolás: konzervatív publikációalapú szabály (egyországos cikk; vagy legalább öt közös cikk egyetlen közös külföldi országgal). Az iteratív jelölteket diagnosztikaként őrizzük, nem fogadjuk el automatikusan.

**Futtatási feltétel:** a 9/D csak teljes, az összes vizsgált intézményt lefedő cache után mehet tovább. A referenciaállományban 5 180 külső intézmény szerepelt. **Az országkapcsolati háló a 22. fejezetben közvetlen publikáció–ország adatokból épül**, nem az intézményi becslésekből.


In [ ]:
publication_institution_link_base, institution_dict_base, institution_link_audit_base = build_publication_institution_link(
    publication_external_root_occurrence=publication_external_root_occurrence,
)

print("publication_institution_link_base alakja:", publication_institution_link_base.shape)
print("institution_dict_base alakja:", institution_dict_base.shape)


In [ ]:
publication_country_occurrence, publication_country_audit = build_publication_country_occurrence(
    df_raw_keyed=df_raw_faculty_publications_keyed,
    pub_id_col="pub_id",
    countries_col="Country/Region",
)
publication_country_audit


In [ ]:
# 9/C – ROR-lekérdezések, folytatható helyi gyorsítótárral
import time
import requests
import pycountry
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROR_CACHE_PATH = INTERIM / "ror_match_cache.csv"
MAX_NEW_QUERIES = None  # Cache nélkül a teljes ROR-lekérdezés lefut; teszt: 100
ROR_DELAY = 0.3

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True
)

institutions = institution_dict_base[["inst_id", "inst_name_canonical"]].copy()
institutions["inst_id"] = institutions["inst_id"].astype(str)

cache_cols = [
    "inst_id", "inst_name_canonical", "ror_status", "ror_id",
    "ror_name", "ror_country_code", "ror_country_name"
]
if ROR_CACHE_PATH.exists():
    ror_cache = pd.read_csv(ROR_CACHE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING, dtype={"inst_id": str})
    ror_cache = ror_cache.reindex(columns=cache_cols)
else:
    ror_cache = pd.DataFrame(columns=cache_cols)

# A cache-bejegyzés csak azonos intézménynév mellett érvényes.
valid_cache = ror_cache.merge(
    institutions, on="inst_id", how="inner", suffixes=("", "_current")
)
valid_cache = valid_cache.loc[
    valid_cache["inst_name_canonical"] == valid_cache["inst_name_canonical_current"]
].drop(columns="inst_name_canonical_current")
valid_cache = valid_cache.drop_duplicates("inst_id", keep="last")
completed = set(valid_cache["inst_id"])
print(f"ROR-cache induláskor: {len(completed)} / {len(institutions)}; hátralévő: {len(institutions) - len(completed)}", flush=True)
todo = institutions.loc[~institutions["inst_id"].isin(completed)]
if MAX_NEW_QUERIES is not None:
    todo = todo.head(MAX_NEW_QUERIES)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": "UniversityResearch-ScopusRORMatching/1.0"})

def save_ror_cache():
    valid_cache.to_csv(ROR_CACHE_PATH, sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

errors = []
for j, item in enumerate(todo.itertuples(index=False), start=1):
    try:
        response = session.get(
            "https://api.ror.org/v2/organizations",
            params={"affiliation": item.inst_name_canonical},
            timeout=(8, 25),
        )
        response.raise_for_status()
        chosen = [x for x in response.json().get("items", []) if x.get("chosen") is True]
        org = chosen[0].get("organization", {}) if len(chosen) == 1 else {}
        names = org.get("names", [])
        display_name = next((n.get("value") for n in names if "ror_display" in n.get("types", [])),
                            names[0].get("value") if names else None)
        locations = org.get("locations", [])
        countries2 = {loc.get("geonames_details", {}).get("country_code") for loc in locations}
        countries2.discard(None)
        c2 = next(iter(countries2)) if len(countries2) == 1 else None
        iso = pycountry.countries.get(alpha_2=c2) if c2 else None
        c3 = iso.alpha_3 if iso else None
        status = "not_chosen" if not chosen else (
            "multi_country" if len(countries2) > 1 else "chosen" if c3 else "no_country"
        )
        record = {
            "inst_id": item.inst_id, "inst_name_canonical": item.inst_name_canonical,
            "ror_status": status, "ror_id": org.get("id"), "ror_name": display_name,
            "ror_country_code": c3,
            "ror_country_name": locations[0].get("geonames_details", {}).get("country_name") if c3 else None,
        }
        valid_cache = pd.concat([valid_cache, pd.DataFrame([record])], ignore_index=True)
        if j % 25 == 0:
            save_ror_cache()
            print(
                f"ROR: {len(valid_cache)}/{len(institutions)} | "
                f"hibák: {len(errors)} | "
                f"utolsó státusz: {status}",
                flush=True
        )
    except (requests.RequestException, ValueError) as exc:
        errors.append((item.inst_id, str(exc)))
        print(f"ROR-hiba [{item.inst_id}]: {exc}", flush=True)

        if len(errors) >= 10:
            save_ror_cache()
            print("Túl sok API-hiba, futás megszakítva.")
            break

    except KeyboardInterrupt:
        save_ror_cache()
        print(f"Futás megszakítva. Mentett rekordok: {len(valid_cache)}")
        raise
save_ror_cache()
print(f"ROR cache: {len(valid_cache)} / {len(institutions)} intézmény")
print("Lekérdezési hibák:", len(errors))
print(valid_cache["ror_status"].value_counts(dropna=False))


In [ ]:
# 9/D – ROR elsődlegesen, konzervatív publikációs módszer másodlagosan
import utils.affiliation_utils as au

assert len(valid_cache) == len(institution_dict_base), (
    "A ROR-cache hiányos. Először fejezd be a 9/C lekérdezéseket."
)

(
    _fallback_pub_inst,
    _fallback_inst,
    fallback_country_dict,
    fallback_audit,
) = infer_institution_countries(
    publication_institution_link=publication_institution_link_base,
    publication_country_occurrence=publication_country_occurrence,
    accept_iterative_estimates=False,
)

# Országok publikációnként, illetve publikációk intézményenként.
country_by_pub = (
    publication_country_occurrence.dropna(subset=["country_code"])
    .groupby("pub_id")["country_code"].agg(lambda s: set(s)).to_dict()
)
pubs_by_inst = (
    publication_institution_link_base.groupby("inst_id")["pub_id"]
    .agg(lambda s: set(s)).to_dict()
)

institution_country_dict = fallback_country_dict.rename(columns={
    "country_code": "fallback_country_code",
    "country": "fallback_country",
    "country_assignment_method": "fallback_method",
}).drop(columns=["is_hungary", "is_resolved"])

institution_country_dict = institution_country_dict.merge(
    valid_cache[cache_cols], on=["inst_id", "inst_name_canonical"], how="left",
    validate="one_to_one"
)
institution_country_dict["country_code"] = institution_country_dict["fallback_country_code"].astype(object)
institution_country_dict["country_assignment_method"] = institution_country_dict["fallback_method"].astype(object)

# A ROR országa akkor automatikusan elfogadható, ha az intézmény minden
# érintett publikációján jelen van. Ellentmondás esetén KÉZI ELLENŐRZÉS.
def count_country_conflicts(row):
    code = row["ror_country_code"]
    if row["ror_status"] != "chosen" or pd.isna(code):
        return pd.NA
    pubs = pubs_by_inst.get(row["inst_id"], set())
    return sum(code not in country_by_pub.get(pub, set()) for pub in pubs)

institution_country_dict["ror_country_conflict_count"] = (
    institution_country_dict.apply(count_country_conflicts, axis=1).astype("Int64")
)
chosen = institution_country_dict["ror_status"].eq("chosen") & institution_country_dict["ror_country_code"].notna()
consistent = chosen & institution_country_dict["ror_country_conflict_count"].eq(0).fillna(False)
conflict = chosen & ~consistent

institution_country_dict.loc[consistent, "country_code"] = institution_country_dict.loc[consistent, "ror_country_code"]
institution_country_dict.loc[consistent, "country_assignment_method"] = "ror_country_consistent"
institution_country_dict.loc[conflict, "country_code"] = None
institution_country_dict.loc[conflict, "country_assignment_method"] = "ror_country_conflict_review"

institution_country_dict["country"] = institution_country_dict["country_code"].map(au.ISO3_TO_COUNTRY)
institution_country_dict["is_hungary"] = institution_country_dict["country_code"].map(
    lambda c: pd.NA if pd.isna(c) else c == "HUN"
).astype("boolean")
institution_country_dict["is_resolved"] = institution_country_dict["country_code"].notna()

publication_institution_link = publication_institution_link_base.merge(
    institution_country_dict[["inst_id", "country", "country_code", "is_hungary",
                              "country_assignment_method", "is_resolved", "ror_id"]],
    on="inst_id", how="left", validate="many_to_one"
)
institution_dict = institution_country_dict.merge(
    institution_dict_base[["inst_id", "n_publications_with_de"]],
    on="inst_id", how="left", validate="one_to_one"
).sort_values("n_publications_with_de", ascending=False, kind="stable").reset_index(drop=True)

institution_link_audit = pd.Series({
    "n_institutions": len(institution_country_dict),
    "n_ror_queried": institution_country_dict["ror_status"].notna().sum(),
    "n_ror_accepted": int(consistent.sum()),
    "n_ror_conflicts": int(conflict.sum()),
    "n_fallback_accepted": int(institution_country_dict["is_resolved"].sum() - consistent.sum()),
    "n_hungarian": int(institution_country_dict["is_hungary"].sum()),
    "n_foreign": int((institution_country_dict["is_hungary"] == False).sum()),
    "n_unresolved": int((~institution_country_dict["is_resolved"]).sum()),
})
assert len(institution_country_dict) == len(institution_dict_base)
assert len(publication_institution_link) == len(publication_institution_link_base)
assert institution_country_dict["inst_id"].is_unique
assert institution_link_audit["n_institutions"] == (
    institution_link_audit["n_hungarian"] +
    institution_link_audit["n_foreign"] +
    institution_link_audit["n_unresolved"]
)
display(institution_link_audit)


In [ ]:
print("ROR-lefedettség:", len(valid_cache), "/", len(institution_dict_base))
print("ORSZÁGBESOROLÁS MÓDSZEREI:")
display(institution_country_dict["country_assignment_method"].fillna("ismeretlen").value_counts())
print("ELLENTMONDÓ ROR-TALÁLATOK – kézi ellenőrzés:")
display(institution_dict.loc[institution_dict["country_assignment_method"].eq("ror_country_conflict_review"),
    ["inst_id", "inst_name_canonical", "ror_id", "ror_name", "ror_country_code",
     "ror_country_conflict_count", "n_publications_with_de"]]
    .sort_values("n_publications_with_de", ascending=False).head(30))
assert len(valid_cache) == len(institution_dict_base), "Hiányos ROR-cache"
assert publication_country_occurrence["country_code"].notna().all(), "Hiányzó ISO-kód"
print("9. fejezet: országkódok és teljes ROR-cache ellenőrizve.")


In [ ]:
publication_institution_link.head(20)


In [ ]:
institution_dict.head(30)


In [ ]:
institution_dict[
    ["inst_id", "inst_name_canonical", "country", "country_code", "country_assignment_method", "n_publications_with_de"]
].head(30)


In [ ]:
unmapped_countries = (
    publication_country_occurrence.loc[
        publication_country_occurrence["country_code"].isna(),
        "country"
    ]
    .dropna()
    .value_counts()
)

unmapped_countries.head(100)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

PUBLICATION_COUNTRY_OCCURRENCE_PATH = INTERIM / "publication_country_occurrence.csv"
INSTITUTION_COUNTRY_DICT_PATH = INTERIM / "institution_country_dict.csv"

publication_country_occurrence.to_csv(
    PUBLICATION_COUNTRY_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

publication_institution_link.to_csv(
    PATHS["publication_institution_link"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

institution_dict.to_csv(
    PATHS["institution_dict"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

institution_country_dict.to_csv(
    INSTITUTION_COUNTRY_DICT_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PUBLICATION_COUNTRY_OCCURRENCE_PATH)
print(" -", PATHS["publication_institution_link"])
print(" -", PATHS["institution_dict"])
print(" -", INSTITUTION_COUNTRY_DICT_PATH)


# 10. Szerző–kar evidencia I. – egykaros publikációk

Egyetlen DE-karhoz kapcsolt publikációban az ID szerint biztos DE-szerző az adott karhoz kap evidenciát. Az egykaros feltétel csökkenti a többkarú cikkekből eredő téves hozzárendelést. Kimenet: `author_kar_evidence_round1` és a hozzá tartozó előfordulások/audit.


In [ ]:
author_kar_occurrence_round1, author_kar_evidence_round1, author_kar_round1_audit = build_author_kar_evidence_round1(
    publication_author_occurrence_deflag=publication_author_occurrence_deflag,
    publication_faculty_link=publication_faculty_link,
    pub_id_col="pub_id",
    kar_col="kar_code",
    de_flag_col="is_de_author_ref_match",
    ref_id_col="matched_ref_scopus_id",
    ref_name_col="matched_ref_name",
)

print("author_kar_occurrence_round1 alakja:", author_kar_occurrence_round1.shape)
print("author_kar_evidence_round1 alakja:", author_kar_evidence_round1.shape)


In [ ]:
author_kar_round1_audit


In [ ]:
author_kar_occurrence_round1.head(20)


In [ ]:
author_kar_evidence_round1.head(20)


In [ ]:
# Hány szerző kapott 1. körben evidence-et?

pd.Series({
    "n_unique_authors_round1": int(author_kar_evidence_round1["ref_scopus_id"].nunique()),
    "n_author_kar_pairs_round1": int(len(author_kar_evidence_round1)),
    "mean_publications_per_author_kar_pair": float(author_kar_evidence_round1["n_publications_round1"].mean()) if len(author_kar_evidence_round1) > 0 else np.nan,
})


In [ ]:
# Hány kar evidence-e jelenik meg az 1. körben?

author_kar_evidence_round1["kar_code"].value_counts().sort_index()


In [ ]:
# Többkarú szerzők már az 1. körben

authors_multi_kar_round1 = (
    author_kar_evidence_round1
    .groupby("ref_scopus_id")["kar_code"]
    .nunique()
    .reset_index(name="n_kars_round1")
    .sort_values("n_kars_round1", ascending=False)
)

authors_multi_kar_round1.head(20)


In [ ]:
# Mely szerzők kapták a legtöbb 1. körös evidence-et?

author_kar_evidence_round1.sort_values(
    ["n_publications_round1", "ref_author_name"],
    ascending=[False, True]
).head(30)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_KAR_OCCURRENCE_R1_PATH = INTERIM / "author_kar_occurrence_round1.csv"
AUTHOR_KAR_EVIDENCE_R1_PATH = PROCESSED / "author_kar_evidence_round1.csv"

author_kar_occurrence_round1.to_csv(
    AUTHOR_KAR_OCCURRENCE_R1_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_kar_evidence_round1.to_csv(
    AUTHOR_KAR_EVIDENCE_R1_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_KAR_OCCURRENCE_R1_PATH)
print(" -", AUTHOR_KAR_EVIDENCE_R1_PATH)


# 11. Szerző–kar evidencia II. – egyértelmű nevek

Csak egykaros, külső intézményt is tartalmazó cikkek. A nem ID-egyező szerző-előfordulás akkor használható, ha a neve a DE-törzsben pontosan egy Scopus ID-hoz tartozik, **és** az adott szerző–kar pár már az első evidenciakörben megjelent. Ez kiegészítő evidencia, nem önálló névalapú DE-szerzőazonosítás.


In [ ]:
author_kar_occurrence_round2, author_kar_evidence_round2, author_kar_round2_audit = build_author_kar_evidence_round2(
    publication_author_occurrence_deflag=publication_author_occurrence_deflag,
    publication_faculty_link=publication_faculty_link,
    publication_institution_link=publication_institution_link,
    de_author_ref=de_author_ref,
    author_kar_evidence_round1=author_kar_evidence_round1,
    pub_id_col="pub_id",
    kar_col="kar_code",
    de_flag_col="is_de_author_ref_match",
    exact_name_flag_col="has_exact_name_in_ref",
    author_name_col="author_name_raw",
    ref_name_col="ref_author_name",
    ref_id_col="ref_scopus_id",
)


In [ ]:
author_kar_round2_audit


In [ ]:
author_kar_occurrence_round2.head(20)


In [ ]:
author_kar_evidence_round2.head(20)


In [ ]:
pd.Series({
    "n_unique_authors_round2": int(author_kar_evidence_round2["ref_scopus_id"].nunique()) if len(author_kar_evidence_round2) > 0 else 0,
    "n_author_kar_pairs_round2": int(len(author_kar_evidence_round2)),
    "mean_publications_per_author_kar_pair_round2": float(author_kar_evidence_round2["n_publications_round2"].mean()) if len(author_kar_evidence_round2) > 0 else np.nan,
})


In [ ]:
author_kar_evidence_round2["kar_code"].value_counts().sort_index()


In [ ]:
authors_multi_kar_round2 = (
    author_kar_evidence_round2
    .groupby("ref_scopus_id")["kar_code"]
    .nunique()
    .reset_index(name="n_kars_round2")
    .sort_values("n_kars_round2", ascending=False)
)

authors_multi_kar_round2.head(20)


In [ ]:
author_kar_evidence_round2.sort_values(
    ["n_publications_round2", "ref_author_name"],
    ascending=[False, True]
).head(30)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_KAR_OCCURRENCE_R2_PATH = INTERIM / "author_kar_occurrence_round2.csv"
AUTHOR_KAR_EVIDENCE_R2_PATH = PROCESSED / "author_kar_evidence_round2.csv"

author_kar_occurrence_round2.to_csv(
    AUTHOR_KAR_OCCURRENCE_R2_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_kar_evidence_round2.to_csv(
    AUTHOR_KAR_EVIDENCE_R2_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_KAR_OCCURRENCE_R2_PATH)
print(" -", AUTHOR_KAR_EVIDENCE_R2_PATH)


# 12. Szerző–kar evidencia III. – többkarú publikációk

Az első két körből egyetlen karhoz lehorgonyozható szerzők adják a támpontot. Többkarú cikkben csak akkor rendelünk új szerzőt karhoz, ha pontosan **egy kar** és **egy biztos, még lehorgonyozatlan DE-szerző** maradt. Bizonytalan helyzetben nincs kényszerbesorolás.


In [ ]:
author_kar_occurrence_round3, author_kar_evidence_round3, author_kar_round3_audit = build_author_kar_evidence_round3(
    publication_author_occurrence_deflag=publication_author_occurrence_deflag,
    publication_faculty_link=publication_faculty_link,
    author_kar_evidence_round1=author_kar_evidence_round1,
    author_kar_evidence_round2=author_kar_evidence_round2,
    pub_id_col="pub_id",
    kar_col="kar_code",
    de_flag_col="is_de_author_ref_match",
    matched_ref_id_col="matched_ref_scopus_id",
    matched_ref_name_col="matched_ref_name",
)

print("author_kar_occurrence_round3 alakja:", author_kar_occurrence_round3.shape)
print("author_kar_evidence_round3 alakja:", author_kar_evidence_round3.shape)


In [ ]:
author_kar_round3_audit


In [ ]:
author_kar_occurrence_round3.head(20)


In [ ]:
author_kar_evidence_round3.head(20)


In [ ]:
pd.Series({
    "n_unique_authors_round3": int(author_kar_evidence_round3["ref_scopus_id"].nunique()) if len(author_kar_evidence_round3) > 0 else 0,
    "n_author_kar_pairs_round3": int(len(author_kar_evidence_round3)),
    "mean_publications_per_author_kar_pair_round3": float(author_kar_evidence_round3["n_publications_round3"].mean()) if len(author_kar_evidence_round3) > 0 else np.nan,
})


In [ ]:
author_kar_evidence_round3["kar_code"].value_counts().sort_index()


In [ ]:
author_kar_evidence_round3.sort_values(
    ["n_publications_round3", "ref_author_name"],
    ascending=[False, True]
).head(30)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_KAR_OCCURRENCE_R3_PATH = INTERIM / "author_kar_occurrence_round3.csv"
AUTHOR_KAR_EVIDENCE_R3_PATH = PROCESSED / "author_kar_evidence_round3.csv"

author_kar_occurrence_round3.to_csv(
    AUTHOR_KAR_OCCURRENCE_R3_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_kar_evidence_round3.to_csv(
    AUTHOR_KAR_EVIDENCE_R3_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_KAR_OCCURRENCE_R3_PATH)
print(" -", AUTHOR_KAR_EVIDENCE_R3_PATH)


# 13. Szerző–kar profil

A három evidenciakör publikációszám-alapú hozzájárulása azonos súllyal kerül összevonásra. `primary_kar`: a legerősebb evidencia karja; döntetlen esetén `INTER`, nulla evidencia esetén hiányzó érték. A teljes kari profil és a hosszú formájú evidenciatábla elkülönül.


In [ ]:
author_kar_profile, author_kar_evidence_long, author_kar_profile_audit = build_author_kar_profile(
    de_author_ref=de_author_ref,
    author_kar_evidence_round1=author_kar_evidence_round1,
    author_kar_evidence_round2=author_kar_evidence_round2,
    author_kar_evidence_round3=author_kar_evidence_round3,
    kar_codes=KAR_CODES,
)

print("author_kar_profile alakja:", author_kar_profile.shape)
print("author_kar_evidence_long alakja:", author_kar_evidence_long.shape)


In [ ]:
author_kar_profile_audit


In [ ]:
author_kar_profile.head(20)


In [ ]:
author_kar_evidence_long.head(20)


In [ ]:
# Primary kar eloszlás

author_kar_profile["primary_kar"].value_counts(dropna=False).sort_index()


In [ ]:
# Hány szerzőnél lett INTER a primary_kar?

author_kar_profile.loc[
    author_kar_profile["primary_kar"] == "INTER",
    ["ref_scopus_id", "ref_author_name", "n_kars_with_evidence", "total_evidence"] + KAR_CODES
].head(20)


In [ ]:
# Hány szerző kapott egyáltalán bármilyen evidence-et?

pd.Series({
    "n_total_de_authors": int(len(author_kar_profile)),
    "n_with_any_evidence": int((author_kar_profile["total_evidence"] > 0).sum()),
    "n_without_evidence": int((author_kar_profile["total_evidence"] <= 0).sum()),
    "share_with_any_evidence": float((author_kar_profile["total_evidence"] > 0).mean()),
})


In [ ]:
# Top szerzők összes evidence szerint

author_kar_profile.sort_values(
    ["total_evidence", "ref_author_name"],
    ascending=[False, True]
).head(30)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_KAR_PROFILE_PATH = PROCESSED / "author_kar_profile.csv"
AUTHOR_KAR_EVIDENCE_LONG_PATH = INTERIM / "author_kar_evidence_long.csv"

author_kar_profile.to_csv(
    AUTHOR_KAR_PROFILE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_kar_evidence_long.to_csv(
    AUTHOR_KAR_EVIDENCE_LONG_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_KAR_PROFILE_PATH)
print(" -", AUTHOR_KAR_EVIDENCE_LONG_PATH)


# 14. Szerzői entitások – konzervatív összevonás

Automatikus összevonás csak azonos név, azonos, nem `INTER` elsődleges kar és **közös publikáció vagy közös DE-társszerző** esetén. A további névcsoportok felülvizsgálati listán maradnak. Több Scopus-profil egy entitássá válhat, de a bibliometriai mutatókhoz **egyetlen reprezentatív forrásprofilt** választunk, nem képezünk összeget.


In [ ]:
author_entity, author_entity_members, author_merge_review, author_entity_audit = build_author_entities(
    de_author_ref=de_author_ref,
    author_kar_profile=author_kar_profile,
    publication_author_occurrence_deflag=publication_author_occurrence_deflag,
    kar_codes=KAR_CODES,
)

print("author_entity alakja:", author_entity.shape)
print("author_entity_members alakja:", author_entity_members.shape)
print("author_merge_review alakja:", author_merge_review.shape)


In [ ]:
source = de_author_ref.set_index("ref_scopus_id")

metric_cols = [
    "citations", "cit_per_pub", "fwci", "h_index",
    "scholarly_output", "most_recent_publication",
    "oldest_publication_since_1996", "primary_affiliation"
]

assert author_entity["metric_source_scopus_id"].isin(source.index).all()

for col in metric_cols:
    actual = author_entity[col]
    expected = author_entity["metric_source_scopus_id"].map(source[col])

    matches = (
        actual.eq(expected) |
        (actual.isna() & expected.isna())
    ).fillna(False)

    assert matches.all(), f"Eltérő profilmetrika: {col}"

print("Minden szerzői metrika a kiválasztott profilból származik.")
print("Többprofilos entitások:",
      (author_entity["n_scopus_ids"] > 1).sum())


In [ ]:
author_entity_audit


In [ ]:
author_entity.head(20)


In [ ]:
author_entity_members.head(20)


In [ ]:
author_merge_review.head(30)


In [ ]:
# Hány entitás több Scopus ID-s?

pd.Series({
    "n_author_entities": int(len(author_entity)),
    "n_multi_scopus_entities": int((author_entity["n_scopus_ids"] > 1).sum()),
    "share_multi_scopus_entities": float((author_entity["n_scopus_ids"] > 1).mean()) if len(author_entity) > 0 else np.nan,
})


In [ ]:
# Több Scopus ID-s entitások mintái

author_entity.loc[
    author_entity["n_scopus_ids"] > 1,
    ["author_id", "canonical_name", "n_scopus_ids", "primary_kar", "total_evidence", "scopus_ids_joined"]
].head(30)


In [ ]:
# Review-t igénylő névcsoportok

author_merge_review["review_reason"].value_counts(dropna=False)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_ENTITY_PATH = PROCESSED / "author_entity.csv"
AUTHOR_ENTITY_MEMBERS_PATH = INTERIM / "author_entity_members.csv"
AUTHOR_MERGE_REVIEW_PATH = INTERIM / "author_merge_review.csv"

author_entity.to_csv(
    AUTHOR_ENTITY_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_entity_members.to_csv(
    AUTHOR_ENTITY_MEMBERS_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_merge_review.to_csv(
    AUTHOR_MERGE_REVIEW_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_ENTITY_PATH)
print(" -", AUTHOR_ENTITY_MEMBERS_PATH)
print(" -", AUTHOR_MERGE_REVIEW_PATH)


# 15. Entitás–Scopus ID kapcsolatok

A `author_id` és a hozzá tartozó Scopus ID-k hosszú táblája a publikációs illesztésekhez kell; a széles tábla az átlátható export. A `(author_id, ref_scopus_id)` párok egyediségét ellenőrizzük.


In [ ]:
author_entity_scopus_ids, author_entity_scopus_ids_long, author_entity_scopus_ids_audit = build_author_entity_scopus_id_tables(
    author_entity=author_entity,
    author_entity_members=author_entity_members,
)

print("author_entity_scopus_ids alakja:", author_entity_scopus_ids.shape)
print("author_entity_scopus_ids_long alakja:", author_entity_scopus_ids_long.shape)


In [ ]:
author_entity_scopus_ids_audit


In [ ]:
author_entity_scopus_ids.head(20)


In [ ]:
author_entity_scopus_ids_long.head(20)


In [ ]:
# Hány Scopus ID oszlop keletkezett a wide táblában?

[c for c in author_entity_scopus_ids.columns if c.startswith("scopus_id_")]


In [ ]:
# Több Scopus ID-s entitások mintái wide formátumban

author_entity_scopus_ids.loc[
    author_entity_scopus_ids["n_scopus_ids"] > 1
].head(30)


In [ ]:
# Ellenőrzés: a long tábla duplikációmentes-e?

author_entity_scopus_ids_long.duplicated(
    subset=["author_id", "ref_scopus_id"]
).sum()


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_ENTITY_SCOPUS_IDS_PATH = PROCESSED / "author_entity_scopus_ids.csv"
AUTHOR_ENTITY_SCOPUS_IDS_LONG_PATH = INTERIM / "author_entity_scopus_ids_long.csv"

author_entity_scopus_ids.to_csv(
    AUTHOR_ENTITY_SCOPUS_IDS_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_entity_scopus_ids_long.to_csv(
    AUTHOR_ENTITY_SCOPUS_IDS_LONG_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_ENTITY_SCOPUS_IDS_PATH)
print(" -", AUTHOR_ENTITY_SCOPUS_IDS_LONG_PATH)


# 16. Publikáció–szerző kapcsolótábla

A biztos DE-szerző-előfordulások az egységes `author_id`-hoz kerülnek. Az egyedi `(pub_id, author_id)` pár egy cikkben csak egyszer számít, még akkor is, ha az eredeti rekord több kari forrásban szerepelt.


In [ ]:
publication_author_entity_link, publication_author_entity_audit = build_publication_author_entity_link(
    publication_author_occurrence_deflag=publication_author_occurrence_deflag,
    author_entity_scopus_ids_long=author_entity_scopus_ids_long,
    pub_id_col="pub_id",
    de_flag_col="is_de_author_ref_match",
    matched_ref_id_col="matched_ref_scopus_id",
    source_kar_col="source_kar",
    source_file_col="source_file",
    author_position_col="author_position",
)

print("publication_author_entity_link alakja:", publication_author_entity_link.shape)


In [ ]:
publication_author_entity_audit


In [ ]:
publication_author_entity_link.head(20)


In [ ]:
# Hány DE author entity jut át publikációnként?

pub_de_author_counts = (
    publication_author_entity_link
    .groupby("pub_id")["author_id"]
    .nunique()
    .reset_index(name="n_de_authors_in_pub")
)

pub_de_author_counts["n_de_authors_in_pub"].describe()


In [ ]:
# Hány publikációban van legalább 1 DE author entity?

pd.Series({
    "n_publications_total": int(publication_master["pub_id"].nunique()),
    "n_publications_with_de_author_entity": int(publication_author_entity_link["pub_id"].nunique()),
    "share_with_de_author_entity": float(
        publication_author_entity_link["pub_id"].nunique() / publication_master["pub_id"].nunique()
    ) if len(publication_master) > 0 else np.nan,
})


In [ ]:
# Ellenőrzés: van-e duplikált pub_id-author_id kapcsolat?

publication_author_entity_link.duplicated(
    subset=["pub_id", "author_id"]
).sum()


In [ ]:
# Top publikációk DE author entity szám szerint

pub_de_author_counts.sort_values(
    ["n_de_authors_in_pub", "pub_id"],
    ascending=[False, True]
).head(30)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

publication_author_entity_link.to_csv(
    PATHS["publication_author_entity_link"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["publication_author_entity_link"])


# 17. DE-s társszerzői háló – élek

Publikációnként minden DE-s szerzőpár élt kap. A nyers közös publikációszám mellett a súly **`1/(k-1)` szerzőpáronként**, ahol `k` a cikkben azonosított DE-s szerzők száma. Ez nem azonos a `1 / C(k,2)` páronkénti felosztással. Önhurok nem kerül az éltáblába.


In [ ]:
author_author_pair_occurrence, author_author_edges, author_author_edges_audit = build_author_author_edges(
    publication_author_entity_link=publication_author_entity_link,
    pub_id_col="pub_id",
    author_id_col="author_id",
    author_position_col="author_position_min",
    weight_mode="inverse_n_minus_1",
)

print("author_author_pair_occurrence alakja:", author_author_pair_occurrence.shape)
print("author_author_edges alakja:", author_author_edges.shape)


In [ ]:
author_author_edges_audit


In [ ]:
author_author_pair_occurrence.head(20)


In [ ]:
author_author_edges.head(20)


In [ ]:
# Hány publikációból lett tényleges DE-DE pár?

pd.Series({
    "n_publications_total": int(publication_master["pub_id"].nunique()),
    "n_publications_with_de_de_pair": int(author_author_pair_occurrence["pub_id"].nunique()) if len(author_author_pair_occurrence) > 0 else 0,
    "share_with_de_de_pair": float(
        author_author_pair_occurrence["pub_id"].nunique() / publication_master["pub_id"].nunique()
    ) if len(publication_master) > 0 else np.nan,
})


In [ ]:
# Él-súlyok eloszlása

author_author_edges["weight"].describe()


In [ ]:
# Közös publikációk száma szerinti eloszlás

author_author_edges["pub_count"].describe()


In [ ]:
# Top élek súly szerint

author_author_edges.sort_values(
    ["weight", "pub_count", "source", "target"],
    ascending=[False, False, True, True]
).head(30)


In [ ]:
# Ellenőrzés: van-e önhurok?

(author_author_edges["source"] == author_author_edges["target"]).sum()


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_AUTHOR_PAIR_OCCURRENCE_PATH = INTERIM / "author_author_pair_occurrence.csv"

author_author_pair_occurrence.to_csv(
    AUTHOR_AUTHOR_PAIR_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

author_author_edges.to_csv(
    PATHS["author_edges_export"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_AUTHOR_PAIR_OCCURRENCE_PATH)
print(" -", PATHS["author_edges_export"])


# 18. Szerzői csomópontok

A szerzői entitásokra publikációszám, bibliometriai adatok és kari evidencia kerül. A Gephi-hez szükséges `Id`/`Label` mezőket itt rögzítjük. A hálózati központisági mutatók a 23. fejezetben egészítik ki a táblát.


In [ ]:
author_nodes_export, author_nodes_audit = build_author_nodes_export(
    author_entity=author_entity,
    publication_author_entity_link=publication_author_entity_link,
    kar_codes=KAR_CODES,
)

print("author_nodes_export alakja:", author_nodes_export.shape)


In [ ]:
author_nodes_audit


In [ ]:
author_nodes_export.head(20)


In [ ]:
# Ellenőrzés: az Id mező egyedi-e?

author_nodes_export["Id"].nunique(), len(author_nodes_export)


In [ ]:
# Primary kar eloszlás

author_nodes_export["primary_kar"].value_counts(dropna=False).sort_index()


In [ ]:
# Multi-Scopus entitások száma

pd.Series({
    "n_author_nodes": int(len(author_nodes_export)),
    "n_multi_scopus_entities": int(author_nodes_export["is_multi_scopus_entity"].sum()),
    "share_multi_scopus_entities": float(author_nodes_export["is_multi_scopus_entity"].mean()) if len(author_nodes_export) > 0 else np.nan,
})


In [ ]:
# Top szerzők publikációszám szerint

author_nodes_export.sort_values(
    ["pub_count_total", "total_evidence", "canonical_name"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Top szerzők publikációszám szerint

author_nodes_export.sort_values(
    ["pub_count_total", "total_evidence", "canonical_name"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Top szerzők h-index szerint

author_nodes_export.sort_values(
    ["h_index", "pub_count_total", "canonical_name"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Kari evidence oszlopok mintája

author_nodes_export[
    ["canonical_name", "primary_kar", "total_evidence", "n_kars_with_evidence"] + KAR_CODES
].head(20)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

author_nodes_export.to_csv(
    PATHS["author_nodes_export"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["author_nodes_export"])


# 19. Kar–kar háló

Egy cikk `k` érintett kara között az egyes különböző karpárok `1/C(k,2)` tört súlyt kapnak; a nyers közös publikációszámot külön tároljuk. Egykaros cikk → önhurok, amely a kari saját publikációk méréséhez használható, **de a hálózati központiságból kimarad**. A `1/(k-1)` alternatív súly a kimenetben szintén elérhető.


In [ ]:
faculty_faculty_pair_occurrence, faculty_faculty_edges, faculty_faculty_edges_audit = build_faculty_faculty_edges(
    publication_faculty_link=publication_faculty_link,
    pub_id_col="pub_id",
    kar_col="kar_code",
)

print("faculty_faculty_pair_occurrence alakja:", faculty_faculty_pair_occurrence.shape)
print("faculty_faculty_edges alakja:", faculty_faculty_edges.shape)


In [ ]:
faculty_faculty_edges_audit


In [ ]:
faculty_faculty_pair_occurrence.head(20)


In [ ]:
faculty_faculty_edges.head(20)


In [ ]:
# Self-loopok száma és aránya

pd.Series({
    "n_edges_total": int(len(faculty_faculty_edges)),
    "n_self_loops": int(faculty_faculty_edges["is_self_loop"].sum()),
    "n_interfaculty_edges": int((~faculty_faculty_edges["is_self_loop"]).sum()),
    "share_self_loops": float(faculty_faculty_edges["is_self_loop"].mean()) if len(faculty_faculty_edges) > 0 else np.nan,
})


In [ ]:
# Nyers publikációszám szerinti top kar-élek

faculty_faculty_edges.sort_values(
    ["pub_count_raw", "weight_frac", "source", "target"],
    ascending=[False, False, True, True]
).head(30)


In [ ]:
# Tört súly szerinti top kar-élek

faculty_faculty_edges.sort_values(
    ["weight_frac", "pub_count_raw", "source", "target"],
    ascending=[False, False, True, True]
).head(30)


In [ ]:
# Inter-faculty élek külön

faculty_faculty_edges.loc[
    ~faculty_faculty_edges["is_self_loop"]
].sort_values(
    ["weight_frac", "pub_count_raw", "source", "target"],
    ascending=[False, False, True, True]
).head(30)


In [ ]:
# Self-loopok külön

faculty_faculty_edges.loc[
    faculty_faculty_edges["is_self_loop"]
].sort_values(
    ["pub_count_raw", "weight_frac", "source"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Ellenőrzés: van-e source-target duplikáció?

faculty_faculty_edges.duplicated(
    subset=["source", "target", "is_self_loop"]
).sum()


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

FACULTY_FACULTY_PAIR_OCCURRENCE_PATH = INTERIM / "faculty_faculty_pair_occurrence.csv"

faculty_faculty_pair_occurrence.to_csv(
    FACULTY_FACULTY_PAIR_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_faculty_edges.to_csv(
    PATHS["faculty_edges_export"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", FACULTY_FACULTY_PAIR_OCCURRENCE_PATH)
print(" -", PATHS["faculty_edges_export"])


# 20. Kari csomópontok

Kar szerinti publikációszám, szerzői evidence és saját publikációt jelző önhurok-mutatók. A `Id`/`Label` oszlopok egységesítik a Gephi-bevitelt; a gráfmetrikák hozzáadása később történik.


In [ ]:
faculty_nodes_export, faculty_nodes_audit = build_faculty_nodes_export(
    faculty_dict=faculty_dict,
    author_kar_profile=author_kar_profile,
    publication_faculty_link=publication_faculty_link,
    faculty_faculty_edges=faculty_faculty_edges,
    kar_codes=KAR_CODES,
)

print("faculty_nodes_export alakja:", faculty_nodes_export.shape)


In [ ]:
faculty_nodes_audit


In [ ]:
faculty_nodes_export


In [ ]:
# Ellenőrzés: az Id mező egyedi-e?

faculty_nodes_export["Id"].nunique(), len(faculty_nodes_export)


In [ ]:
# Top karok publikációszám szerint

faculty_nodes_export.sort_values(
    ["pub_count_total", "kar_code"],
    ascending=[False, True]
)


In [ ]:
# Top karok csak ide sorolt szerzők száma szerint

faculty_nodes_export.sort_values(
    ["n_authors_primary_only", "kar_code"],
    ascending=[False, True]
)


In [ ]:
# Top karok bármilyen evidence-es szerzőszám szerint

faculty_nodes_export.sort_values(
    ["n_authors_with_any_evidence", "kar_code"],
    ascending=[False, True]
)


In [ ]:
# Self-loop mezők sanity check

faculty_nodes_export[
    ["kar_code", "self_loop_pub_count_raw", "self_loop_weight_frac", "self_loop_weight_inv_n_minus_1"]
].sort_values(
    ["self_loop_pub_count_raw", "kar_code"],
    ascending=[False, True]
)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

faculty_nodes_export.to_csv(
    PATHS["faculty_nodes_export"],
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PATHS["faculty_nodes_export"])


# 21. DE–intézmény és kar–intézmény élek

Egy publikáció `m` külső intézménye DE-szinten fejenként `1/m` tört súlyt kap. `k` DE-kar esetén a kar–intézmény párok súlya `1/(k·m)`. A nyers közös publikációszám külön mező, ezért a két mérték nem összekeverhető. Intézményi országcímke kizárólag a 9/D elfogadott vagy tartalék besorolásából származik.


In [ ]:
# Intézményi országcímke: a 9/D ellenőrzött besorolása.
assert "n_ror_accepted" in institution_link_audit.index, "Futtasd a 9/D cellát!"
assert len(publication_institution_link) == len(publication_institution_link_base)

(
    publication_de_institution_occurrence,
    de_institution_edges,
    publication_faculty_institution_occurrence,
    faculty_institution_edges,
    de_institution_audit,
) = build_de_institution_edges(
    publication_faculty_link=publication_faculty_link,
    publication_institution_link=publication_institution_link,
)

print("publication_de_institution_occurrence alakja:", publication_de_institution_occurrence.shape)
print("de_institution_edges alakja:", de_institution_edges.shape)
print("publication_faculty_institution_occurrence alakja:", publication_faculty_institution_occurrence.shape)
print("faculty_institution_edges alakja:", faculty_institution_edges.shape)
# A 21. fejezet auditjának egyeznie kell a 9/D intézményi besorolásával.
expected_hu = int(institution_link_audit["n_hungarian"])
expected_foreign = int(institution_link_audit["n_foreign"])
assert de_institution_audit["n_hungarian_external_institutions"] == expected_hu, (
    "Régi intézményország-adat került a 21. fejezetbe!"
)
assert de_institution_audit["n_nonhungarian_external_institutions"] == expected_foreign, (
    "Régi intézményország-adat került a 21. fejezetbe!"
)
assert de_institution_audit["n_de_institution_edges"] == len(institution_dict_base)
print(f"ROR-alapú intézményháló ellenőrizve: HU={expected_hu}, külföld={expected_foreign}, ismeretlen={int(institution_link_audit['n_unresolved'])}")


In [ ]:
de_institution_audit


In [ ]:
publication_de_institution_occurrence.head(20)


In [ ]:
de_institution_edges.head(30)


In [ ]:
publication_faculty_institution_occurrence.head(20)


In [ ]:
faculty_institution_edges.head(30)


In [ ]:
# Teljes DE legfontosabb partnerei tört súly szerint

de_institution_edges.sort_values(
    ["weight_frac", "pub_count_raw", "inst_name_canonical"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Magyar partnerek

de_institution_edges.loc[
    de_institution_edges["is_hungary"] == True
].sort_values(
    ["weight_frac", "pub_count_raw", "inst_name_canonical"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Nem magyar partnerek

de_institution_edges.loc[
    de_institution_edges["is_hungary"] == False
].sort_values(
    ["weight_frac", "pub_count_raw", "inst_name_canonical"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Egy adott kar legfontosabb partnerei példa (IK)

faculty_institution_edges.loc[
    faculty_institution_edges["kar_code"] == "IK"
].sort_values(
    ["weight_frac", "pub_count_raw", "inst_name_canonical"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Duplikációellenőrzés

pd.Series({
    "dup_de_inst_edges": int(
        de_institution_edges.duplicated(subset=["inst_id"]).sum()
    ),
    "dup_fac_inst_edges": int(
        faculty_institution_edges.duplicated(subset=["kar_code", "inst_id"]).sum()
    ),
})


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

PUBLICATION_DE_INSTITUTION_OCCURRENCE_PATH = INTERIM / "publication_de_institution_occurrence.csv"
PUBLICATION_FACULTY_INSTITUTION_OCCURRENCE_PATH = INTERIM / "publication_faculty_institution_occurrence.csv"
DE_INSTITUTION_EDGES_PATH = PROCESSED / "de_institution_edges.csv"
FACULTY_INSTITUTION_EDGES_PATH = PROCESSED / "faculty_institution_edges.csv"

publication_de_institution_occurrence.to_csv(
    PUBLICATION_DE_INSTITUTION_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

de_institution_edges.to_csv(
    DE_INSTITUTION_EDGES_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

publication_faculty_institution_occurrence.to_csv(
    PUBLICATION_FACULTY_INSTITUTION_OCCURRENCE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_institution_edges.to_csv(
    FACULTY_INSTITUTION_EDGES_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", PUBLICATION_DE_INSTITUTION_OCCURRENCE_PATH)
print(" -", DE_INSTITUTION_EDGES_PATH)
print(" -", PUBLICATION_FACULTY_INSTITUTION_OCCURRENCE_PATH)
print(" -", FACULTY_INSTITUTION_EDGES_PATH)


# 22. DE–ország és kar–ország háló

A `Country/Region` publikációs mezőből számolunk, **nem** a ROR-os intézményi országcímkéből. Magyarország nem külső partnerország. Ha egy cikkhez `m` különböző külföldi ország és `k` DE-kar tartozik, akkor a DE–ország súly `1/m`, a kar–ország súly `1/(k·m)`. Mindkét szinten a tört súlyok összege a nemzetközi publikációk száma; ezt assert ellenőrzi.


In [ ]:
# A 0. fejezetből importált build_country_aggregations függvényt használjuk.
# utils/network_utils.py országaggregációja publikációszintű bemenetet vár.
de_country_edges, faculty_country_edges, country_aggregation_audit = build_country_aggregations(
    publication_country_occurrence=publication_country_occurrence,
    publication_faculty_link=publication_faculty_link,
)

print(country_aggregation_audit.to_string())

print("de_country_edges alakja:", de_country_edges.shape)
print("faculty_country_edges alakja:", faculty_country_edges.shape)


In [ ]:
country_aggregation_audit


In [ ]:
de_country_edges.head(30)


In [ ]:
faculty_country_edges.head(30)


In [ ]:
# Teljes DE legfontosabb országpartnerei

de_country_edges.sort_values(
    ["weight_frac", "pub_count_raw", "country"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Karonkénti országkapcsolatok példa: ÁOK

faculty_country_edges.loc[
    faculty_country_edges["kar_code"] == "ÁOK"
].sort_values(
    ["weight_frac", "pub_count_raw", "country"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Duplikációellenőrzés

pd.Series({
    "dup_de_country_edges": int(
        de_country_edges.duplicated(subset=["country_code"]).sum()
    ),
    "dup_fac_country_edges": int(
        faculty_country_edges.duplicated(subset=["kar_code", "country_code"]).sum()
    ),
})


In [ ]:
n_foreign_pubs = publication_country_occurrence.loc[
    publication_country_occurrence["country_code"].notna()
    & publication_country_occurrence["country_code"].ne("HUN"),
    "pub_id"
].nunique()

assert not de_country_edges["country_code"].eq("HUN").any()
assert de_country_edges["country_code"].is_unique
assert not faculty_country_edges.duplicated(
    ["kar_code", "country_code"]
).any()

assert abs(de_country_edges["weight_frac"].sum() - n_foreign_pubs) < 1e-6
assert abs(faculty_country_edges["weight_frac"].sum() - n_foreign_pubs) < 1e-6

print("Külföldi országok:", len(de_country_edges))
print("Kar–ország kapcsolatok:", len(faculty_country_edges))
print("Nemzetközi publikációk:", n_foreign_pubs)
print("Minden ellenőrzés sikeres.")
assert n_foreign_pubs == 3808, "A vizsgált publikációhalmaz eltér a várt mintától."
assert len(de_country_edges) == 145, "A külföldi országok száma eltér."
assert len(faculty_country_edges) == 838, "A kar–ország kapcsolatok száma eltér."


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

DE_COUNTRY_EDGES_PATH = PROCESSED / "de_country_edges.csv"
FACULTY_COUNTRY_EDGES_PATH = PROCESSED / "faculty_country_edges.csv"

de_country_edges.to_csv(
    DE_COUNTRY_EDGES_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_country_edges.to_csv(
    FACULTY_COUNTRY_EDGES_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", DE_COUNTRY_EDGES_PATH)
print(" -", FACULTY_COUNTRY_EDGES_PATH)


# 23. Gráfmetrikák

**Szerzőháló:** fokszám, súlyozott fokszám (strength), súlyozott PageRank, komponensméret és komponensazonosító. A súlyozatlan és súlyozott betweenness **500 véletlen forráscsomóponttal közelített**, rögzített seeddel (a háló méretétől függően legfeljebb az összes csomópont).

**Kari háló:** ugyanezek a fő metrikák, a betweenness azonban teljes számítással készül; a self-loopok nem kerülnek a számított gráfba. A súly mint kapcsolaterősség, a legrövidebb utakhoz használt távolság pedig `1/weight`.


## 23/A. Szerzőháló – közelített központiságok


In [ ]:
cached_author_metrics = COMPUTE_CACHE.cache(
    compute_author_network_metrics
)

author_nodes_export_enriched, author_network_metrics_audit = cached_author_metrics(
    author_nodes_export=author_nodes_export,
    author_author_edges=author_author_edges,
    node_id_col="author_id",
    weight_col="weight",
    betweenness_k=500,
    random_seed=42,
)

print("author_nodes_export_enriched alakja:", author_nodes_export_enriched.shape)


In [ ]:
author_network_metrics_audit


In [ ]:
author_nodes_export_enriched.head(20)


In [ ]:
# Top szerzők weighted PageRank szerint

author_nodes_export_enriched.sort_values(
    ["author_pagerank_weighted", "author_strength", "canonical_name"],
    ascending=[False, False, True]
).head(30)


In [ ]:
# Komponenseloszlás

author_nodes_export_enriched["author_component_size"].describe()


## 23/B. Kari háló – egzakt központiságok


In [ ]:
faculty_nodes_export_enriched, faculty_network_metrics_audit = compute_faculty_network_metrics(
    faculty_nodes_export=faculty_nodes_export,
    faculty_faculty_edges=faculty_faculty_edges,
    node_id_col="kar_code",
    weight_col="weight_frac",
)

print("faculty_nodes_export_enriched alakja:", faculty_nodes_export_enriched.shape)


In [ ]:
faculty_network_metrics_audit


In [ ]:
faculty_nodes_export_enriched


In [ ]:
# Top karok weighted PageRank szerint

faculty_nodes_export_enriched.sort_values(
    ["faculty_pagerank_weighted", "faculty_strength_weight_frac", "kar_code"],
    ascending=[False, False, True]
)


In [ ]:
# Top karok weighted betweenness szerint

faculty_nodes_export_enriched.sort_values(
    ["faculty_betweenness_weighted", "faculty_strength_weight_frac", "kar_code"],
    ascending=[False, False, True]
)


In [ ]:
# Komponensek

faculty_nodes_export_enriched[[
    "kar_code",
    "faculty_component_id",
    "faculty_component_size",
    "faculty_degree",
    "faculty_strength_weight_frac"
]].sort_values(
    ["faculty_component_id", "kar_code"],
    ascending=[True, True]
)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása.

AUTHOR_NODES_ENRICHED_PATH = PROCESSED / "author_nodes_export_enriched.csv"
FACULTY_NODES_ENRICHED_PATH = PROCESSED / "faculty_nodes_export_enriched.csv"

author_nodes_export_enriched.to_csv(
    AUTHOR_NODES_ENRICHED_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_nodes_export_enriched.to_csv(
    FACULTY_NODES_ENRICHED_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_NODES_ENRICHED_PATH)
print(" -", FACULTY_NODES_ENRICHED_PATH)


# 24. Gephi-kompatibilis exportok

Külön szerzői, kari, külső intézményi és országkapcsolati csúcs-/éltáblák. `Id`/`Label` a csomópontoknál; `Source`/`Target`/`Weight` az éleknél. A `Weight` jelentése hálózattípustól függ; a **nyers** és **tört** súlyokat nem szabad egy skálán összemosni. A 24. fejezet GEXF 1.3-ban előállítja a hazai intézményi csillagot és a nemzetközi ország–intézmény hierarchikus nézet alapját. A Gephi-beli végső elrendezés külön lépés.


In [ ]:
(
    author_nodes_gephi,
    author_edges_gephi,
    faculty_nodes_gephi,
    faculty_edges_gephi,
    institution_nodes_gephi,
    de_institution_edges_gephi,
    faculty_institution_edges_gephi,
    de_country_edges_gephi,
    faculty_country_edges_gephi,
    gephi_export_audit,
) = build_gephi_exports(
    author_nodes_export_enriched=author_nodes_export_enriched,
    author_author_edges=author_author_edges,
    faculty_nodes_export_enriched=faculty_nodes_export_enriched,
    faculty_faculty_edges=faculty_faculty_edges,
    institution_dict=institution_dict,
    de_institution_edges=de_institution_edges,
    faculty_institution_edges=faculty_institution_edges,
    de_country_edges=de_country_edges,
    faculty_country_edges=faculty_country_edges,
)

print("author_nodes_gephi alakja:", author_nodes_gephi.shape)
print("author_edges_gephi alakja:", author_edges_gephi.shape)
print("faculty_nodes_gephi alakja:", faculty_nodes_gephi.shape)
print("faculty_edges_gephi alakja:", faculty_edges_gephi.shape)
print("institution_nodes_gephi alakja:", institution_nodes_gephi.shape)


In [ ]:
gephi_export_audit


In [ ]:
# Gephi-kritikus ellenőrzések

pd.Series({
    "author_nodes_unique_id": int(author_nodes_gephi["Id"].nunique() == len(author_nodes_gephi)),
    "faculty_nodes_unique_id": int(faculty_nodes_gephi["Id"].nunique() == len(faculty_nodes_gephi)),
    "institution_nodes_unique_id": int(institution_nodes_gephi["Id"].nunique() == len(institution_nodes_gephi)),
    "author_edges_missing_source": int(author_edges_gephi["Source"].isna().sum()),
    "author_edges_missing_target": int(author_edges_gephi["Target"].isna().sum()),
    "faculty_edges_missing_source": int(faculty_edges_gephi["Source"].isna().sum()),
    "faculty_edges_missing_target": int(faculty_edges_gephi["Target"].isna().sum()),
})


In [ ]:
author_nodes_gephi.head(10)


In [ ]:
author_edges_gephi.head(10)


In [ ]:
faculty_nodes_gephi.head(10)


In [ ]:
faculty_edges_gephi.head(10)


In [ ]:
institution_nodes_gephi.head(10)


In [ ]:
de_country_edges_gephi.head(10)


In [ ]:
# Ellenőrző és feldolgozott állományok kiírása. Gephi mappába

GEPHI_DIR = PROCESSED / "gephi"
ensure_directories([GEPHI_DIR])

AUTHOR_NODES_GEPHI_PATH = GEPHI_DIR / "author_nodes_gephi.csv"
AUTHOR_EDGES_GEPHI_PATH = GEPHI_DIR / "author_edges_gephi.csv"

FACULTY_NODES_GEPHI_PATH = GEPHI_DIR / "faculty_nodes_gephi.csv"
FACULTY_EDGES_GEPHI_PATH = GEPHI_DIR / "faculty_edges_gephi.csv"

INSTITUTION_NODES_GEPHI_PATH = GEPHI_DIR / "institution_nodes_gephi.csv"
DE_INSTITUTION_EDGES_GEPHI_PATH = GEPHI_DIR / "de_institution_edges_gephi.csv"
FACULTY_INSTITUTION_EDGES_GEPHI_PATH = GEPHI_DIR / "faculty_institution_edges_gephi.csv"

DE_COUNTRY_EDGES_GEPHI_PATH = GEPHI_DIR / "de_country_edges_gephi.csv"
FACULTY_COUNTRY_EDGES_GEPHI_PATH = GEPHI_DIR / "faculty_country_edges_gephi.csv"

author_nodes_gephi.to_csv(AUTHOR_NODES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)
author_edges_gephi.to_csv(AUTHOR_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)

faculty_nodes_gephi.to_csv(FACULTY_NODES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)
faculty_edges_gephi.to_csv(FACULTY_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)

institution_nodes_gephi.to_csv(INSTITUTION_NODES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)
de_institution_edges_gephi.to_csv(DE_INSTITUTION_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)
faculty_institution_edges_gephi.to_csv(FACULTY_INSTITUTION_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)

de_country_edges_gephi.to_csv(DE_COUNTRY_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)
faculty_country_edges_gephi.to_csv(FACULTY_COUNTRY_EDGES_GEPHI_PATH, index=SAVE_INDEX, encoding=CSV_ENCODING, sep=CSV_SEP, quoting=CSV_QUOTING)

print("Mentve a Gephi exportok:")
print(" -", AUTHOR_NODES_GEPHI_PATH)
print(" -", AUTHOR_EDGES_GEPHI_PATH)
print(" -", FACULTY_NODES_GEPHI_PATH)
print(" -", FACULTY_EDGES_GEPHI_PATH)
print(" -", INSTITUTION_NODES_GEPHI_PATH)
print(" -", DE_INSTITUTION_EDGES_GEPHI_PATH)
print(" -", FACULTY_INSTITUTION_EDGES_GEPHI_PATH)
print(" -", DE_COUNTRY_EDGES_GEPHI_PATH)
print(" -", FACULTY_COUNTRY_EDGES_GEPHI_PATH)


In [ ]:
import math
import textwrap
from pathlib import Path
import pandas as pd
import networkx as nx

TOP_HU = 35
TOP_COUNTRIES = 8
TOP_INST_PER_COUNTRY = 8
INST_LABEL_MAX = 39   # Ennyi karakter körül rövidül a megjelenített intézménynév.
OUT = Path(GEPHI_DIR) / "intezmenyi_nezetek"
OUT.mkdir(parents=True, exist_ok=True)

inst = institution_nodes_gephi.copy()
links = de_institution_edges_gephi.copy()
countries = de_country_edges_gephi.copy()
for df, col in [(inst, "Id"), (links, "Target")]:
    df[col] = df[col].astype(str)
assert inst["Id"].is_unique and links["Target"].is_unique
assert set(links["Target"]) <= set(inst["Id"]), "Hiányzó intézményi csomópont."

inst = inst.merge(
    links[["Target", "Weight", "pub_count_raw"]],
    left_on="Id", right_on="Target", how="inner", validate="one_to_one"
).drop(columns="Target")
inst["pub_count_raw"] = pd.to_numeric(inst["pub_count_raw"], errors="raise")
inst["Weight"] = pd.to_numeric(inst["Weight"], errors="raise")
inst = inst.sort_values(
    ["pub_count_raw", "Weight", "Id"],
    ascending=[False, False, True], kind="stable"
).reset_index(drop=True)

# Jól elkülöníthető, színvaksággal is többé-kevésbé kompatibilis színskála.
COLORS = ["#4477AA", "#EE6677", "#228833", "#CCBB44",
          "#AA3377", "#66CCEE", "#AA7744", "#999933"]

def rgba(hex_color, alpha=1.0):
    h = hex_color.lstrip("#")
    return {"r": int(h[0:2], 16), "g": int(h[2:4], 16),
            "b": int(h[4:6], 16), "a": float(alpha)}

def pos(x, y):
    return {"x": float(x), "y": float(y), "z": 0.0}

def nice_label(name):
    name = " ".join(str(name).split())
    return textwrap.shorten(name, width=INST_LABEL_MAX, placeholder="…")

def base_graph(center_label="DE"):
    G = nx.Graph()
    G.add_node(
        "DE", label=center_label, FullName="Debreceni Egyetem",
        NodeType="university", CountryGroup="HUN", CountryName="Hungary",
        PubCount=0, FractionalWeight=0.0, LabelSize=18.0,
        viz={"position": pos(0, 0), "size": 90.0,
             "color": rgba("#20304F")},
    )
    return G

def add_institution(G, row, x, y, color, parent, max_weight):
    inst_id = str(row["Id"])
    name = str(row["inst_name_canonical"])
    w = float(row["Weight"])
    pubs = int(row["pub_count_raw"])
    G.add_node(
        inst_id, label=nice_label(name), FullName=name,
        NodeType="institution", CountryGroup=str(row["country_code"]),
        CountryName=str(row["country"]),
        PubCount=pubs, FractionalWeight=w, LabelSize=10.0,
        viz={"position": pos(x, y),
             "size": float(8 + 3.8 * math.log1p(pubs)),
             "color": rgba(color)},
    )
    # Ország--intézmény: az intézménynek a DE-vel közös publikációi.
    thickness = 0.65 + 1.9 * math.sqrt(w / max(max_weight, 1e-9))
    G.add_edge(
        parent, inst_id, weight=w, PubCount=pubs,
        RelationType="country_institution" if parent != "DE" else "DE_institution",
        WeightMethod="institution_fractional_publications",
        viz={"thickness": float(thickness), "color": rgba(color, 0.57)},
    )

# A. HAZAI: megtartjuk a DE-központú csillagot; minden csomópont kap címkét.
G_hu = base_graph(center_label="DE")
hu = inst.loc[inst["country_code"].eq("HUN")].head(TOP_HU).copy()
max_hu_w = float(hu["Weight"].max()) if len(hu) else 1.0
for i, (_, row) in enumerate(hu.iterrows()):
    theta = i * math.pi * (3 - math.sqrt(5))
    radius = 290 + 95 * math.sqrt(i)
    add_institution(
        G_hu, row, radius * math.cos(theta), radius * math.sin(theta),
        "#2B9C91", "DE", max_hu_w,
    )

# B. NEMZETKÖZI: DE -> ISO3 országcsomópont -> országonként legyező intézmények.
# A DE--ország él közvetlen publikáció--ország aggregációból származik,
# az ország--intézmény él intézményi adatokból. E súlyok nem összeadhatók!
known_cc = set(inst["country_code"].dropna().astype(str)) - {"HUN"}
country_rank = (
    countries.loc[
        countries["country_code"].isin(known_cc) &
        countries["country_code"].ne("HUN")
    ].sort_values(["Weight", "country_code"], ascending=[False, True], kind="stable")
)
top_countries = country_rank["country_code"].drop_duplicates().head(TOP_COUNTRIES).tolist()
assert len(top_countries) >= 6, "Hatnál kevesebb azonosítható partnerország."

G_world = base_graph(center_label="DE")
country_rows = []
selected_parts = []
max_country_w = float(country_rank["Weight"].max())
max_inst_w = float(inst["Weight"].max())
n_country = len(top_countries)
R_COUNTRY = 770       # DE -> ország távolság
R_INST_INNER = 1780   # DE -> intézmények radiális alaptávolsága
R_INST_OUTER = 1970   # Válogatott intézmények második gyűrűje
HALF_FAN = math.radians(17.3)  # 8 csoportnál 45 fokos szektorok

for j, cc in enumerate(top_countries):
    cr = country_rank.loc[country_rank["country_code"].eq(cc)].iloc[0]
    color = COLORS[j % len(COLORS)]
    theta = -math.pi / 2 + 2 * math.pi * j / n_country
    country_id = f"COUNTRY_{cc}"  # Nem ütközik a Scopus intézmény-ID-kkel.
    country_name = str(cr["country"])
    country_pubs = int(cr["pub_count_raw"])
    country_weight = float(cr["Weight"])

    G_world.add_node(
        country_id, label=cc, FullName=country_name,
        NodeType="country", CountryGroup=cc, CountryName=country_name,
        PubCount=country_pubs, FractionalWeight=country_weight, LabelSize=15.0,
        viz={"position": pos(R_COUNTRY * math.cos(theta),
                              R_COUNTRY * math.sin(theta)),
             "size": 57.0, "color": rgba(color)},
    )
    # Az országok közötti élszélesség az országaggregáció súlya alapján változik.
    G_world.add_edge(
        "DE", country_id, weight=country_weight,
        PubCount=country_pubs, RelationType="DE_country",
        WeightMethod="country_fractional_publications",
        viz={"thickness": float(2.2 + 5.0 * math.sqrt(
            country_weight / max(max_country_w, 1e-9))),
             "color": rgba(color, 0.85)},
    )

    part = inst.loc[inst["country_code"].eq(cc)].head(TOP_INST_PER_COUNTRY).copy()
    part["MapCountryRank"] = j + 1
    selected_parts.append(part)
    count = len(part)
    assert count >= 1
    offsets = [0.0] if count == 1 else [
        -HALF_FAN + k * 2 * HALF_FAN / (count - 1)
        for k in range(count)
    ]

    # Széles szögű legyező: a levelek külön irányban indulnak az országcsomópontból.
    # Az eltolt radiális gyűrűk segítenek a hosszú címkék elkülönítésében.
    for k, (_, row) in enumerate(part.iterrows()):
        angle = theta + offsets[k]
        radius = R_INST_INNER if k % 2 == 0 else R_INST_OUTER
        add_institution(
            G_world, row, radius * math.cos(angle), radius * math.sin(angle),
            color, country_id, max_inst_w,
        )

    country_rows.append({
        "CountryCode": cc, "CountryName": country_name,
        "CountryPubCount": country_pubs, "CountryFractionalWeight": country_weight,
        "ShownInstitutions": count,
        "ShownInstitutionPubCountSum_NOT_UNIQUE": int(part["pub_count_raw"].sum()),
    })

# C. ELLENŐRZÉS ÉS GEXF 1.3 EXPORT
assert G_hu.number_of_nodes() == 1 + len(hu)
assert G_hu.number_of_edges() == len(hu)
assert G_world.number_of_nodes() == 1 + n_country + sum(len(p) for p in selected_parts)
assert G_world.number_of_edges() == n_country + sum(len(p) for p in selected_parts)
assert all(G_world.has_edge("DE", f"COUNTRY_{cc}") for cc in top_countries)
assert all(
    G_world.nodes[v]["NodeType"] == "country"
    for v in G_world.neighbors("DE")
)

for G, filename in [
    (G_hu, "01_DE_hazai_top_partnerek_v2.gexf"),
    (G_world, "02_DE_nemzetkozi_orszag_intezmeny_fa_v2.gexf"),
]:
    path = OUT / filename
    nx.write_gexf(G, path, version="1.3", encoding="utf-8")
    loaded = nx.read_gexf(path, version="1.3")
    assert loaded.number_of_nodes() == G.number_of_nodes()
    assert loaded.number_of_edges() == G.number_of_edges()

hu.to_csv(OUT / "01_hazai_kivalasztott_intezmenyek.csv",
          index=False, sep=";", encoding="utf-8-sig")
pd.concat(selected_parts, ignore_index=True).to_csv(
    OUT / "02_nemzetkozi_kivalasztott_intezmenyek.csv",
    index=False, sep=";", encoding="utf-8-sig",
)
pd.DataFrame(country_rows).to_csv(
    OUT / "02_nemzetkozi_orszag_osszesites.csv",
    index=False, sep=";", encoding="utf-8-sig",
)
print(f"Hazai háló: {G_hu.number_of_nodes()} csomópont, {G_hu.number_of_edges()} él")
print(f"Nemzetközi háló: {G_world.number_of_nodes()} csomópont, {G_world.number_of_edges()} él")
print("Országok:", ", ".join(top_countries))
print("Fájltípus: GEXF 1.3 | Mentés:", OUT)


# 25. Összehasonlító statisztikák és ábrák

**11 egységes stílusú ábra:** kari hálózati ujjlenyomat, k-mag-görbék, óriáskomponens-arány és medián h-index, kari oszlopdiagramok, hat mintakar szerzői szórásdiagramjai, külföldi országpartnerek és kari PageRank. Kimenet: `data_out/processed/figures/ch25_meta/` (300 DPI PNG + vektoros SVG), illetve a `processed/meta/` összesítések.

**Értelmezés:** a h-index és a vizsgált mintabeli publikációszám `x=y` vonala csak a mintán kívüli publikációs teljesítmény szükségességét jelzi; nem bizonyít más affiliációt. A nem értelmezhető asszortativitás `NaN`, nem nulla. A kari óriáskomponens-arány és medián h-index két külön mérőszám, nem összevont kompozit index.


In [ ]:
# 25. fejezet – közös ábrastílus, kari kódolás, mérőszámok

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Kimeneti mappa
CH25_FIG_DIR = PROCESSED / "figures" / "ch25_meta"
CH25_DATA_DIR = PROCESSED / "meta"
ensure_directories([CH25_FIG_DIR, CH25_DATA_DIR])

# Egységes kari színek
# A KK-KGyC a klinikai központ csoportszínének világosabb árnyalatát kapja.

FACULTY_COLORS = {
    "ÁOK": "#558DFF",
    "IK": "#B3D91A",
    "ÁJK": "#A255FF",
    "BTK": "#923000",
    "MÉTKK": "#FFB358",
    "MÉKK": "#FFB358",      # belső alias
    "GTK": "#DD87FF",
    "FOK": "#7DDCFF",
    "GyTK": "#72FFF0",
    "GYTK": "#72FFF0",      # belső alias
    "TTK": "#2DBF00",
    "AKT": "#FFCF13",
    "MK": "#FA7100",
    "NEK": "#759AFF",
    "KK": "#7093FF",
    "KK-KGyC": "#9DB7FF",   # kiegészítő szín
    "EÜK": "#85AFFF",
    "ZMK": "#A26300",
    "GYFK": "#EC78FF",
    "GyFK": "#EC78FF",      # belső alias
}

# 6 minta kar
# Ugyanez a hat kar szerepel a két szerzői szórásdiagramon.
SAMPLE_FACULTIES = ["ÁOK", "IK", "GTK", "MK", "MÉKK", "TTK"]

# Segédfüggvények
def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Nincs meg egyik oszlop sem: {candidates}")

def robust_z(series):
    """Robusztus z-érték: a hiányzó értékeket nem tekinti nullának."""
    s = pd.to_numeric(series, errors="coerce")
    valid = s.dropna()
    if valid.empty:
        return s
    med = valid.median()
    mad = np.median(np.abs(valid - med))
    if not np.isfinite(mad) or mad == 0:
        std = valid.std(ddof=0)
        if not np.isfinite(std) or std == 0:
            return pd.Series(0.0, index=s.index).where(s.notna())
        return (s - valid.mean()) / std
    return (s - med) / (1.4826 * mad)

def weighted_gini(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x >= 0]
    if len(x) == 0:
        return 0.0
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

def safe_density(g):
    n = g.number_of_nodes()
    if n <= 1:
        return 0.0
    return nx.density(g)

def safe_avg_clustering(g, weight="weight"):
    if g.number_of_nodes() <= 1:
        return 0.0
    try:
        return nx.average_clustering(g, weight=weight)
    except Exception:
        return 0.0

def safe_assortativity(g, weight="weight"):
    """Súlyozott fokszám-asszortativitás; nem értelmezhető esetben NaN."""
    if g.number_of_edges() == 0 or g.number_of_nodes() < 2:
        return np.nan
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            val = nx.degree_assortativity_coefficient(g, weight=weight)
        return float(val) if np.isfinite(val) else np.nan
    except Exception:
        return np.nan

def strength_dict(g, weight="weight"):
    return dict(g.degree(weight=weight))

def top10_strength_share(g, weight="weight"):
    sdict = strength_dict(g, weight=weight)
    vals = np.array(list(sdict.values()), dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0 or np.allclose(vals.sum(), 0):
        return 0.0
    vals = np.sort(vals)[::-1]
    top_n = max(1, int(np.ceil(len(vals) * 0.10)))
    return float(vals[:top_n].sum() / vals.sum())

def giant_component_fraction(g):
    n = g.number_of_nodes()
    if n == 0:
        return 0.0
    comps = list(nx.connected_components(g))
    if len(comps) == 0:
        return 0.0
    giant = max(len(c) for c in comps)
    return giant / n

def kcore_signature(g):
    if g.number_of_nodes() == 0 or g.number_of_edges() == 0:
        return {}, {}
    core_num = nx.core_number(g)
    max_k = max(core_num.values()) if len(core_num) else 0
    frac = {}
    for k in range(1, max_k + 1):
        frac[k] = np.mean([v >= k for v in core_num.values()])
    return core_num, frac

def canonical_faculty_code(x):
    if pd.isna(x):
        return x
    if x == "MÉTKK":
        return "MÉKK"
    return x

def faculty_display_label(x):
    if pd.isna(x):
        return x
    if x == "MÉKK":
        return "MÉKK"
    return x

def faculty_color(x):
    if pd.isna(x):
        return "#888888"
    return FACULTY_COLORS.get(x, "#888888")

# 25. fejezet: egységes megjelenítési stílus (ábrák 1–11)
from matplotlib.colors import TwoSlopeNorm

CH25_STYLE = {
    "ink": "#243447",
    "muted": "#617080",
    "grid": "#DDE4EC",
    "border": "#D3DBE5",
    "country": "#547FA5",  # nem kari kategória
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",  # magyar ékezetek
    "font.size": 10.5,
    "axes.titlesize": 15,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.labelcolor": CH25_STYLE["ink"],
    "text.color": CH25_STYLE["ink"],
    "xtick.color": CH25_STYLE["muted"],
    "ytick.color": CH25_STYLE["muted"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": CH25_STYLE["border"],
    "axes.linewidth": 0.8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 300,
    "svg.fonttype": "none",  # SVG-nél a betűk szerkeszthetők
})

def ch25_axes(ax, *, grid_axis="y"):
    """Közös tengely- és rácsstílus, az adatértékek megváltoztatása nélkül."""
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color=CH25_STYLE["grid"], linewidth=0.8, alpha=0.85)
    ax.spines["left"].set_color(CH25_STYLE["border"])
    ax.spines["bottom"].set_color(CH25_STYLE["border"])
    ax.tick_params(axis="both", length=3.5, width=0.8)

def ch25_save(fig, filename):
    """Azonos, jó minőségű PNG és vektoros SVG export."""
    path = CH25_FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(path.with_suffix(".svg"), bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

def ch25_faculty_bars(data, value_col, title, xlabel, filename, *, number_format=".0f"):
    """A kari összehasonlítások egységes vízszintes oszlopdiagramja."""
    sub = data.copy()
    sub["kar_label_plot"] = sub["kar_code"].map(faculty_display_label)
    sub = sub.sort_values([value_col, "kar_code"], ascending=[False, True])
    sub[value_col] = pd.to_numeric(sub[value_col], errors="coerce")

    fig, ax = plt.subplots(figsize=(10.5, max(6.8, 0.40 * len(sub) + 1.9)))
    bars = ax.barh(
        sub["kar_label_plot"], sub[value_col], height=0.70,
        color=[faculty_color(k) for k in sub["kar_code"]],
        edgecolor=CH25_STYLE["ink"], linewidth=0.4, zorder=3,
    )
    ax.invert_yaxis()
    ax.set_title(title, loc="left", pad=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ch25_axes(ax, grid_axis="x")

    vmax = float(sub[value_col].max()) if len(sub) else 0
    if np.isfinite(vmax) and vmax > 0:
        ax.set_xlim(0, vmax * 1.20)
        for bar, v in zip(bars, sub[value_col]):
            if pd.notna(v):
                label = format(float(v), number_format).replace(".", ",")
                ax.text(bar.get_width() + vmax * 0.012,
                        bar.get_y() + bar.get_height()/2,
                        label, va="center", ha="left", fontsize=9,
                        color=CH25_STYLE["muted"])
    fig.tight_layout()
    ch25_save(fig, filename)

# Alapadatok
authors_meta_base = prepare_author_meta_analysis_base(
    author_nodes_export_enriched=author_nodes_export_enriched
).copy()

faculty_meta_summary = build_faculty_meta_summary(
    author_nodes_export_enriched=author_nodes_export_enriched,
    faculty_nodes_export_enriched=faculty_nodes_export_enriched,
    faculty_country_edges=faculty_country_edges,
).copy()

# Kari kódok egységesítése
authors_meta_base["primary_kar"] = authors_meta_base["primary_kar"].map(canonical_faculty_code)
faculty_meta_summary["kar_code"] = faculty_meta_summary["kar_code"].map(canonical_faculty_code)
faculty_nodes_export_enriched["kar_code"] = faculty_nodes_export_enriched["kar_code"].map(canonical_faculty_code)
faculty_country_edges["kar_code"] = faculty_country_edges["kar_code"].map(canonical_faculty_code)

# Oszlopok felderítése
AUTHOR_ID_COL = first_existing(authors_meta_base, ["author_id", "Id"])
AUTHOR_LABEL_COL = first_existing(authors_meta_base, ["canonical_name", "Label", "ref_author_name"])
AUTHOR_KAR_COL = first_existing(authors_meta_base, ["primary_kar"])
PUB_COL = first_existing(authors_meta_base, ["pub_count_total", "pub_count"])
H_COL = first_existing(authors_meta_base, ["h_index"])
FWCI_COL = first_existing(authors_meta_base, ["fwci"])
STRENGTH_COL = first_existing(authors_meta_base, ["author_strength"])

EDGE_SOURCE_COL = first_existing(author_author_edges, ["source", "Source"])
EDGE_TARGET_COL = first_existing(author_author_edges, ["target", "Target"])
EDGE_WEIGHT_COL = first_existing(author_author_edges, ["weight", "Weight"])

# Teljes DE szerzői gráf
G = nx.Graph()
valid_author_ids = set(authors_meta_base[AUTHOR_ID_COL].dropna().astype(str))

# Minden szerző felvétele, az izolált csomópontokkal együtt
G.add_nodes_from(valid_author_ids)

for _, row in author_author_edges.iterrows():
    s = str(row[EDGE_SOURCE_COL])
    t = str(row[EDGE_TARGET_COL])
    w = pd.to_numeric(row[EDGE_WEIGHT_COL], errors="coerce")

    if s not in valid_author_ids or t not in valid_author_ids:
        continue
    if pd.isna(w) or w <= 0 or s == t:
        continue

    G.add_edge(s, t, weight=float(w))

# Kari részgráfok
faculty_author_map = (
    authors_meta_base[[AUTHOR_ID_COL, AUTHOR_KAR_COL]]
    .dropna(subset=[AUTHOR_ID_COL, AUTHOR_KAR_COL])
    .groupby(AUTHOR_KAR_COL)[AUTHOR_ID_COL]
    .apply(lambda s: sorted(set(s.astype(str))))
    .to_dict()
)

faculty_subgraphs = {}
for kar, nodes in faculty_author_map.items():
    sub_nodes = [n for n in nodes if n in G]
    faculty_subgraphs[kar] = G.subgraph(sub_nodes).copy()

# Kari sorrend
preferred_order = [
    "ÁOK", "IK", "ÁJK", "BTK", "MÉKK", "GTK", "FOK", "GyTK",
    "TTK", "AKT", "MK", "NEK", "KK", "KK-KGyC", "EÜK", "ZMK", "GYFK"
]
existing_order = [k for k in preferred_order if k in set(faculty_nodes_export_enriched["kar_code"])]
remaining_order = [k for k in sorted(set(faculty_nodes_export_enriched["kar_code"])) if k not in existing_order]
FACULTY_ORDER = existing_order + remaining_order

# Alapaudit
chapter25_audit_base = pd.Series({
    "n_authors_meta_base": int(len(authors_meta_base)),
    "n_faculties_meta_summary": int(len(faculty_meta_summary)),
    "n_sample_faculties": int(len(SAMPLE_FACULTIES)),
    "n_faculty_subgraphs": int(len(faculty_subgraphs)),
    "n_author_graph_nodes": int(G.number_of_nodes()),
    "n_author_graph_edges": int(G.number_of_edges()),
})
assert G.number_of_nodes() == 7288
assert G.number_of_edges() == 40176
assert nx.number_of_isolates(G) == 2561

print("A szerzői gráf javítása sikeres.")
chapter25_audit_base


In [ ]:
# 1. ábra – Kari hálózati ujjlenyomat / hőtérkép

fingerprint_rows = []

for kar in FACULTY_ORDER:
    sg = faculty_subgraphs.get(kar, nx.Graph())

    n = sg.number_of_nodes()
    m = sg.number_of_edges()

    sdict = strength_dict(sg, weight="weight")
    strengths = list(sdict.values())

    n_components = nx.number_connected_components(sg) if n > 0 else 0
    giant_frac = giant_component_fraction(sg)
    core_num, core_frac = kcore_signature(sg)

    row = {
        "kar_code": kar,
        "density": safe_density(sg),
        "n_components": n_components,
        "gini_strength": weighted_gini(strengths),
        "avg_clustering": safe_avg_clustering(sg, weight="weight"),
        "k_max": max(core_num.values()) if len(core_num) else 0,
        "core_frac_k>=5": core_frac.get(5, 0.0),
        "giant_frac": giant_frac,
        "core_frac_k>=3": core_frac.get(3, 0.0),
        "median_strength": float(np.median(strengths)) if len(strengths) else 0.0,
        "assortativity": safe_assortativity(sg, weight="weight"),
        "top10_strength_share": top10_strength_share(sg, weight="weight"),
        "n_nodes": n,
        "n_edges": m,
    }
    fingerprint_rows.append(row)

faculty_fingerprint_df = pd.DataFrame(fingerprint_rows)

heatmap_metric_order = [
    "density",
    "n_components",
    "gini_strength",
    "avg_clustering",
    "k_max",
    "core_frac_k>=5",
    "giant_frac",
    "core_frac_k>=3",
    "median_strength",
    "assortativity",
    "top10_strength_share",
]

heatmap_df = faculty_fingerprint_df[["kar_code"] + heatmap_metric_order].copy()

for col in heatmap_metric_order:
    heatmap_df[col] = robust_z(heatmap_df[col])

heatmap_df["kar_code"] = pd.Categorical(
    heatmap_df["kar_code"],
    categories=FACULTY_ORDER,
    ordered=True
)
heatmap_df = heatmap_df.sort_values("kar_code").copy()
heatmap_df["kar_label_plot"] = heatmap_df["kar_code"].astype(str).map(faculty_display_label)

plot_matrix = heatmap_df[heatmap_metric_order].to_numpy()

# Metrikánkénti robusztus standardizálás; nulla a választott középérték.
valid = plot_matrix[np.isfinite(plot_matrix)]
lim = max(2.0, float(np.nanpercentile(np.abs(valid), 95))) if len(valid) else 2.0
cmap = plt.colormaps["RdBu_r"].copy()
cmap.set_bad("#ECEFF3")

METRIC_LABELS = {
    "density": "Sűrűség", "n_components": "Komponensek száma",
    "gini_strength": "Strength Gini", "avg_clustering": "Klaszterezettség",
    "k_max": "Max. k-mag", "core_frac_k>=5": "5-mag aránya",
    "giant_frac": "Óriáskomponens", "core_frac_k>=3": "3-mag aránya",
    "median_strength": "Medián strength", "assortativity": "Asszortativitás",
    "top10_strength_share": "Felső 10% strength-aránya",
}

fig, ax = plt.subplots(figsize=(14, 9.5))
im = ax.imshow(
    np.ma.masked_invalid(plot_matrix), aspect="auto", cmap=cmap,
    norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim),
)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Robusztus z-érték (metrikánként)")
ax.set_yticks(range(len(heatmap_df)))
ax.set_yticklabels(heatmap_df["kar_label_plot"], fontsize=10)
ax.set_xticks(range(len(heatmap_metric_order)))
ax.set_xticklabels(
    [METRIC_LABELS[x] for x in heatmap_metric_order],
    rotation=38, ha="right", fontsize=9,
)
ax.set_title("Kari hálózati ujjlenyomat", loc="left", pad=14)
ax.tick_params(axis="both", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
ch25_save(fig, "fig_25_01_kari_halozati_ujjlenyomat_heatmap.png")


In [ ]:
# 2. ábra – Kari k-mag szignatúragörbék

kcore_rows = []

for kar in FACULTY_ORDER:
    sg = faculty_subgraphs.get(kar, nx.Graph())
    _, frac_dict = kcore_signature(sg)
    for k, frac in frac_dict.items():
        kcore_rows.append({
            "kar_code": kar,
            "k": k,
            "frac_in_kcore": frac
        })

faculty_kcore_df = pd.DataFrame(kcore_rows)

fig, ax = plt.subplots(figsize=(12.8, 7.5))
for kar in FACULTY_ORDER:
    sub = faculty_kcore_df.loc[
        faculty_kcore_df["kar_code"] == kar
    ].sort_values("k")
    if sub.empty:
        continue
    # A hat mintakar hangsúlyosabb; a többi kar továbbra is látható.
    is_sample = kar in SAMPLE_FACULTIES
    ax.plot(
        sub["k"], sub["frac_in_kcore"],
        lw=2.7 if is_sample else 1.4,
        alpha=1.0 if is_sample else 0.72,
        label=faculty_display_label(kar), color=faculty_color(kar),
    )
ax.set_title("Kari k-mag szignatúragörbék", loc="left", pad=14)
ax.set_xlabel("k-mag index")
ax.set_ylabel("A legalább k-maghoz tartozó szerzők aránya")
ax.set_ylim(0, 1.04)
ch25_axes(ax, grid_axis="both")
ax.legend(
    title="Kar", bbox_to_anchor=(1.02, 1), loc="upper left",
    fontsize=9, title_fontsize=10, frameon=False, ncol=2,
    borderaxespad=0, handlelength=2.5,
)
fig.tight_layout()
ch25_save(fig, "fig_25_02_kmag_szignaturagorbek.png")


In [ ]:
# 3. ábra – Kari hálózati összekapcsoltság és medián h-index
# Vízszintes tengely: óriáskomponens-arány; függőleges: kari medián h-index.

from matplotlib.lines import Line2D

faculty_connectivity_hindex_df = faculty_fingerprint_df[
    ["kar_code", "giant_frac", "n_nodes", "n_edges"]
].merge(
    faculty_meta_summary[["kar_code", "h_index_median"]],
    on="kar_code", how="left", validate="one_to_one"
)

label_offsets = {
    "GTK": (7, -14), "EÜK": (7, 9),
    "GyTK": (-39, 12), "MÉKK": (7, -11),
    "ZMK": (-38, 7), "ÁOK": (7, 0),
}

fig, ax = plt.subplots(figsize=(12, 7.8))
for _, row in faculty_connectivity_hindex_df.iterrows():
    x = row["giant_frac"]
    y = row["h_index_median"]
    n = int(row["n_nodes"])
    kar = row["kar_code"]
    if not (np.isfinite(x) and np.isfinite(y)):
        continue

    small = n < 20
    color = faculty_color(kar)
    ax.scatter(
        x, y,
        s=50 + 10 * np.sqrt(n),
        facecolors="none" if small else color,
        edgecolors=color if small else CH25_STYLE["ink"],
        linewidths=1.1, alpha=0.90, zorder=3,
    )
    ax.annotate(
        faculty_display_label(kar), (x, y),
        xytext=label_offsets.get(kar, (6, 5)),
        textcoords="offset points", fontsize=9,
        color=CH25_STYLE["ink"],
        bbox=dict(facecolor="white", alpha=0.70, edgecolor="none", pad=0.25),
    )

ax.set_xlim(-0.05, 1.07)
ax.set_ylim(bottom=-0.15)
ax.set_title("Kari hálózati összekapcsoltság és medián h-index", loc="left", pad=14)
ax.set_xlabel("Óriáskomponensbe tartozó aktív szerzők aránya")
ax.set_ylabel("Medián h-index (Scopus)")
ch25_axes(ax, grid_axis="both")
ax.legend(
    handles=[Line2D([], [], marker="o", markerfacecolor="none",
                    markeredgecolor=CH25_STYLE["muted"], linestyle="None",
                    label="Kis hálózat (<20 aktív csomópont)")],
    loc="upper left", fontsize=9,
)
fig.tight_layout()
ch25_save(fig, "fig_25_03_integracio_median_hindex.png")


In [ ]:
# 4. ábra – Karok publikációszáma
ch25_faculty_bars(
    faculty_meta_summary, "pub_count_total",
    "Karok publikációszáma", "Egyedi publikációk száma",
    "fig_25_04_karok_publikacioszama.png", number_format=".0f",
)


In [ ]:
# 5. ábra – Karok medián h-indexe
ch25_faculty_bars(
    faculty_meta_summary, "h_index_median",
    "Karok medián h-indexe", "Medián h-index",
    "fig_25_05_karok_median_hindex.png", number_format=".1f",
)


In [ ]:
# 6. ábra – Karok medián szerzői strength értéke
ch25_faculty_bars(
    faculty_meta_summary, "author_strength_median",
    "Karok medián szerzői strength értéke", "Medián szerzői strength",
    "fig_25_06_karok_median_strength.png", number_format=".2f",
)


In [ ]:
# 7. ábra – Publikációszám vs h-index, 6 minta kar, 2x3 panel
# 0 publikációs esetek kiszűrése

plot7_df = authors_meta_base[[AUTHOR_KAR_COL, PUB_COL, H_COL]].copy()
plot7_df[AUTHOR_KAR_COL] = plot7_df[AUTHOR_KAR_COL].map(canonical_faculty_code)
plot7_df[PUB_COL] = pd.to_numeric(plot7_df[PUB_COL], errors="coerce")
plot7_df[H_COL] = pd.to_numeric(plot7_df[H_COL], errors="coerce")

plot7_df = plot7_df[
    plot7_df[AUTHOR_KAR_COL].isin(SAMPLE_FACULTIES) &
    plot7_df[PUB_COL].notna() &
    plot7_df[H_COL].notna() &
    (plot7_df[PUB_COL] > 0)
].copy()

xmax = plot7_df[PUB_COL].max()
ymax = plot7_df[H_COL].max()

# Hat panel, azonos tengelyhatárokkal; x=y referencia.
fig, axes = plt.subplots(3, 2, figsize=(12.5, 13.4), sharex=True, sharey=True)
axes = axes.ravel()
xlim = max(float(xmax) * 1.04, 1)
ylim = max(float(ymax) * 1.04, 1)
line_max = min(xlim, ylim)

for ax, kar in zip(axes, SAMPLE_FACULTIES):
    sub = plot7_df.loc[plot7_df[AUTHOR_KAR_COL] == kar]
    ax.plot([0, line_max], [0, line_max],
            ls="--", lw=1.3, color=CH25_STYLE["ink"], alpha=0.75, zorder=2)
    if not sub.empty:
        ax.scatter(sub[PUB_COL], sub[H_COL], s=22, alpha=0.62,
                   color=faculty_color(kar), edgecolors="white",
                   linewidths=0.25, zorder=3)
    else:
        ax.text(0.5, 0.5, "Nincs adat", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(faculty_display_label(kar), fontsize=12, loc="left", pad=9)
    ax.set_xlim(0, xlim)
    ax.set_ylim(0, ylim)
    ch25_axes(ax, grid_axis="both")

fig.suptitle("Szerzői publikációszám és h-index – hat mintakar", fontsize=16, y=0.995)
fig.supxlabel("Publikációk száma a vizsgált kari mintában", y=0.058, fontsize=11)
fig.supylabel("Scopus h-index", x=0.012, fontsize=11)
fig.text(0.5, 0.022, "Szaggatott vonal: h-index = mintabeli publikációszám; fölötte további publikációk szükségesek a h-indexhez.",
         ha="center", va="center", fontsize=9, color=CH25_STYLE["muted"])
fig.tight_layout(rect=[0.03, 0.105, 0.99, 0.97])
ch25_save(fig, "fig_25_07_pubcount_hindex_mintakarok.png")


In [ ]:
# 8. ábra – Szerzői strength vs FWCI, 6 minta kar, 2x3 panel
# y tengely logaritmikus

plot8_df = authors_meta_base[[AUTHOR_KAR_COL, STRENGTH_COL, FWCI_COL]].copy()
plot8_df[AUTHOR_KAR_COL] = plot8_df[AUTHOR_KAR_COL].map(canonical_faculty_code)
plot8_df[STRENGTH_COL] = pd.to_numeric(plot8_df[STRENGTH_COL], errors="coerce")
plot8_df[FWCI_COL] = pd.to_numeric(plot8_df[FWCI_COL], errors="coerce")

plot8_df = plot8_df[
    plot8_df[AUTHOR_KAR_COL].isin(SAMPLE_FACULTIES) &
    plot8_df[STRENGTH_COL].notna() &
    plot8_df[FWCI_COL].notna() &
    (plot8_df[STRENGTH_COL] > 0) &
    (plot8_df[FWCI_COL] > 0)
].copy()

xmax = plot8_df[STRENGTH_COL].max()
ymin = plot8_df[FWCI_COL].min()
ymax = plot8_df[FWCI_COL].max()
ymin_plot = 10 ** np.floor(np.log10(ymin))
ymax_plot = 10 ** np.ceil(np.log10(ymax))

fig, axes = plt.subplots(3, 2, figsize=(12.5, 13.4), sharex=True, sharey=True)
axes = axes.ravel()

for ax, kar in zip(axes, SAMPLE_FACULTIES):
    sub = plot8_df.loc[plot8_df[AUTHOR_KAR_COL] == kar]
    if not sub.empty:
        ax.scatter(sub[STRENGTH_COL], sub[FWCI_COL], s=22, alpha=0.62,
                   color=faculty_color(kar), edgecolors="white",
                   linewidths=0.25, zorder=3)
    else:
        ax.text(0.5, 0.5, "Nincs adat", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(faculty_display_label(kar), fontsize=12, loc="left", pad=9)
    ax.set_yscale("log")
    ax.set_xlim(0, max(float(xmax)*1.04, 1))
    ax.set_ylim(ymin_plot, ymax_plot)
    if ymin_plot <= 1 <= ymax_plot:
        ax.axhline(1, lw=1.2, ls="--", color=CH25_STYLE["ink"], alpha=0.7, zorder=2)
    ch25_axes(ax, grid_axis="both")

fig.suptitle("Szerzői strength és FWCI – hat mintakar", fontsize=16, y=0.995)
fig.supxlabel("Szerzői strength", y=0.058, fontsize=11)
fig.supylabel("FWCI (logaritmikus skála)", x=0.012, fontsize=11)
fig.text(0.5, 0.022, "A szaggatott vonal az FWCI = 1 referenciaértéket jelöli (ha a tengelytartományba esik).",
         ha="center", va="center", fontsize=9, color=CH25_STYLE["muted"])
fig.tight_layout(rect=[0.03, 0.105, 0.99, 0.97])
ch25_save(fig, "fig_25_08_strength_fwci_mintakarok_log.png")


In [ ]:
# 9. ábra – DE legfontosabb nemzetközi országpartnerei
# Országaggregáció: semleges szín, nem kari kategória.

plot_df = de_country_edges.loc[
    de_country_edges["is_hungary"] == False
].copy()

plot_df["country_label"] = plot_df["country"].fillna(plot_df["country_code"])
plot_df = plot_df.sort_values(
    ["weight_frac", "pub_count_raw", "country_label"],
    ascending=[False, False, True]
).head(15)

fig, ax = plt.subplots(figsize=(11, 7.7))
bars = ax.barh(plot_df["country_label"], plot_df["weight_frac"],
               color=CH25_STYLE["country"], height=0.72, zorder=3)
ax.invert_yaxis()  # legnagyobb partner felül
ax.set_title("A Debreceni Egyetem legfontosabb külföldi országpartnerei", loc="left", pad=14)
ax.set_xlabel("Tört publikációs hozzájárulás")
ax.set_ylabel("")
ch25_axes(ax, grid_axis="x")
max_v = float(plot_df["weight_frac"].max()) if len(plot_df) else 0
if max_v > 0:
    ax.set_xlim(0, max_v * 1.18)
    for bar, val in zip(bars, plot_df["weight_frac"]):
        ax.text(bar.get_width()+max_v*0.012, bar.get_y()+bar.get_height()/2,
                f"{val:.1f}".replace(".", ","),
                va="center", color=CH25_STYLE["muted"], fontsize=9)
fig.tight_layout()
ch25_save(fig, "fig_25_09_de_top_nemzetkozi_orszagpartnerek.png")


In [ ]:
# 10. ábra – Külföldi partnerországok száma karonként
ch25_faculty_bars(
    faculty_meta_summary, "n_nonhungary_country_partners",
    "Külföldi partnerországok száma karonként", "Külföldi országok száma",
    "fig_25_10_nemmagyar_orszagpartnerek_karonkent.png", number_format=".0f",
)


In [ ]:
# 11. ábra – Karok súlyozott PageRank értéke
ch25_faculty_bars(
    faculty_nodes_export_enriched, "faculty_pagerank_weighted",
    "Karok súlyozott PageRank értéke", "Súlyozott PageRank",
    "fig_25_11_karok_weighted_pagerank.png", number_format=".4f",
)


In [ ]:
# A 25. fejezet összesítőinek és auditjának mentése.

FACULTY_META_SUMMARY_PATH = CH25_DATA_DIR / "faculty_meta_summary.csv"
AUTHORS_META_BASE_PATH = CH25_DATA_DIR / "authors_meta_base.csv"
FACULTY_FINGERPRINT_PATH = CH25_DATA_DIR / "faculty_fingerprint_df.csv"
FACULTY_CONNECTIVITY_HINDEX_PATH = CH25_DATA_DIR / "faculty_integration_hindex.csv"

faculty_meta_summary.to_csv(
    FACULTY_META_SUMMARY_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

authors_meta_base.to_csv(
    AUTHORS_META_BASE_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_fingerprint_df.to_csv(
    FACULTY_FINGERPRINT_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

faculty_connectivity_hindex_df.to_csv(
    FACULTY_CONNECTIVITY_HINDEX_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

meta_analysis_audit = pd.Series({
    "n_authors_meta_base": int(len(authors_meta_base)),
    "n_faculties_meta_summary": int(len(faculty_meta_summary)),
    "n_faculties_in_fingerprint": int(len(faculty_fingerprint_df)),
    "n_faculties_in_connectivity_hindex": int(len(faculty_connectivity_hindex_df)),
    "n_sample_faculties": int(len(SAMPLE_FACULTIES)),
    "n_figures_saved": 11,
    "sample_faculties": " | ".join([faculty_display_label(k) for k in SAMPLE_FACULTIES]),
    "figures_dir": str(CH25_FIG_DIR),
})

print("Mentve:")
print(" -", FACULTY_META_SUMMARY_PATH)
print(" -", AUTHORS_META_BASE_PATH)
print(" -", FACULTY_FINGERPRINT_PATH)
print(" -", FACULTY_CONNECTIVITY_HINDEX_PATH)

meta_analysis_audit


# 26. KAT – szerzői kategóriák karon belül

Két dimenzió: **súlyozott betweenness** a szerző saját karának indukált részgráfján, valamint **Scopus h-index**. A csoporthatárok karonként Q1/Q3 kvartilisek; az azonos értékek nem választhatók szét. Ha Q1 = Q3, csak a határnál szigorúan nagyobb érték kerülhet felülre. Nulla betweenness-variancia vagy egyéb hiányzó feltétel esetén a szerző **nem besorolható**. Az eredmény a Gephi-szerzőtáblában egy `KAT` oszloppal egészül ki; a részletes ellenőrzés külön állomány.

**Fontos:** ez **karhoz viszonyított kategorizálás**, nem abszolút karok közötti teljesítményrangsor.


In [ ]:
import importlib
import utils.analysis_utils as analysis_utils

importlib.reload(analysis_utils)
build_author_nodes_gephi_kat = analysis_utils.build_author_nodes_gephi_kat

cached_kat = COMPUTE_CACHE.cache(
    build_author_nodes_gephi_kat
)

author_nodes_gephi_KAT, kat_details, kat_faculty_summary, kat_audit = cached_kat(
    author_nodes_gephi=author_nodes_gephi,
    author_edges_gephi=author_edges_gephi,
    min_faculty_size=8,
)

print("author_nodes_gephi_KAT alakja:", author_nodes_gephi_KAT.shape)
print("kat_details alakja:", kat_details.shape)
print("kat_faculty_summary alakja:", kat_faculty_summary.shape)


In [ ]:
kat_audit


In [ ]:
author_nodes_gephi_KAT.head(20)


In [ ]:
kat_details.head(20)


In [ ]:
kat_details["KAT"].value_counts(dropna=False)


In [ ]:
kat_faculty_summary


In [ ]:
# Top "sztárok"
kat_details.loc[
    kat_details["KAT"] == "sztárok"
].sort_values(
    ["sci_metric_h_index", "embed_metric_local_betweenness"],
    ascending=[False, True]
).head(30)


In [ ]:
# Top "integrátorok"
kat_details.loc[
    kat_details["KAT"] == "integrátorok"
].sort_values(
    ["embed_metric_local_betweenness", "sci_metric_h_index"],
    ascending=[False, True]
).head(30)


In [ ]:
AUTHOR_NODES_GEPHI_KAT_PATH = PROCESSED / "gephi" / "author_nodes_gephi_KAT.csv"
KAT_DETAILS_PATH = PROCESSED / "kat_details.csv"
KAT_FACULTY_SUMMARY_PATH = PROCESSED / "kat_faculty_summary.csv"

author_nodes_gephi_KAT.to_csv(
    AUTHOR_NODES_GEPHI_KAT_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

kat_details.to_csv(
    KAT_DETAILS_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

kat_faculty_summary.to_csv(
    KAT_FACULTY_SUMMARY_PATH,
    index=SAVE_INDEX,
    encoding=CSV_ENCODING,
    sep=CSV_SEP,
    quoting=CSV_QUOTING
)

print("Mentve:")
print(" -", AUTHOR_NODES_GEPHI_KAT_PATH)
print(" -", KAT_DETAILS_PATH)
print(" -", KAT_FACULTY_SUMMARY_PATH)
